<a href="https://colab.research.google.com/github/Praharshita1275/Criminal_mind_analysis/blob/main/Criminal_mind_analysis_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-Processing


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## LLM1


In [5]:
import pandas as pd
import numpy as np


In [6]:
#load dataset

df = pd.read_csv("/content/Crime_Data_from_2020_to_Present.csv")

df.head()
df.info()
df.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 326977 entries, 0 to 326976
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   DATE OCC      326977 non-null  object 
 1   TIME OCC      326977 non-null  int64  
 2   AREA NAME     326977 non-null  object 
 3   Vict Age      326977 non-null  int64  
 4   Vict Sex      326977 non-null  object 
 5   Vict Descent  326977 non-null  object 
 6   Premis Desc   326977 non-null  object 
 7   Weapon Desc   326977 non-null  object 
 8   Status Desc   326977 non-null  object 
 9   LOCATION      326977 non-null  object 
 10  LAT           326977 non-null  float64
 11  LON           326977 non-null  float64
 12  Crm Cd Desc   326977 non-null  object 
dtypes: float64(2), int64(2), object(9)
memory usage: 32.4+ MB


,0
DATE OCC,0
TIME OCC,0
AREA NAME,0
Vict Age,0
Vict Sex,0
Vict Descent,0
Premis Desc,0
Weapon Desc,0
Status Desc,0
LOCATION,0


In [7]:

#standardise column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.columns


Index(['date_occ', 'time_occ', 'area_name', 'vict_age', 'vict_sex',
       'vict_descent', 'premis_desc', 'weapon_desc', 'status_desc', 'location',
       'lat', 'lon', 'crm_cd_desc'],
      dtype='object')

In [8]:
#HANDLE MISSING & INVALID VALUES
text_cols = [
    "area_name", "premis_desc", "weapon_desc",
    "status_desc", "crm_cd_desc", "location"
]

for col in text_cols:
    df[col] = df[col].fillna("UNKNOWN")

df["vict_age"] = df["vict_age"].replace(0, np.nan)
df["vict_age"] = df["vict_age"].fillna("UNKNOWN")



In [9]:
# BASIC TEXT CLEANING (LIGHT)
def clean_text(col):
    return (
        col.astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

for col in text_cols:
    df[col] = clean_text(df[col])


In [10]:
# FIX DATE & TIME
df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")

def time_to_hour(x):
    try:
        x = int(x)
        return x // 100
    except:
        return "UNKNOWN"

df["time_hour"] = df["time_occ"].apply(time_to_hour)


/tmp/ipython-input-1543/1132658626.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")


In [11]:
# SELECT ONLY USEFUL COLUMNS
keep_cols = [
    "date_occ", "time_hour", "area_name",
    "vict_age", "vict_sex",
    "premis_desc", "weapon_desc",
    "status_desc", "crm_cd_desc",
    "location"
]

df = df[keep_cols]



In [50]:
# CREATE THE MOST IMPORTANT COLUMN (crime_text)
# ─────────────────────────────────────────────────────────────────────────────
# FIX: crm_cd_desc is INTENTIONALLY removed from crime_text.
#
# WHY: The old version included the crime type name (e.g. "robbery") directly
# in the text, AND used it to create the label (robbery → financial).
# The model just memorized: see "robbery" → predict "financial" → 100% accuracy.
# That's circular — not generalization.
#
# NOW: The model must infer motivation from CONTEXT only:
# time, location, weapon, victim age/sex, status — not the crime type name.
# ─────────────────────────────────────────────────────────────────────────────

def create_crime_text(row):
    return (
        f"On {row['date_occ']} at {row['time_hour']} hours, "
        f"in {row['area_name']} area, a {row['vict_age']}-year-old "
        f"{row['vict_sex']} was involved in an incident "
        f"at {row['premis_desc']}. "
        f"Weapon used: {row['weapon_desc']}. "
        f"Case status: {row['status_desc']}."
    )

df["crime_text"] = df.apply(create_crime_text, axis=1)

print("✅ crime_text created (crm_cd_desc excluded — bias fix applied)")
print("Sample:")
print(df["crime_text"].iloc[0])


KeyError: 'date_occ'

In [13]:
df["crime_text"].head(3)


,crime_text
0,"On 2020-05-10 at 22 hours, in central area, a ..."
1,"On 2020-12-02 at 22 hours, in 77th street area..."
2,"On 2020-05-01 at 23 hours, in 77th street area..."


In [14]:
#ADD SIMPLE RULE-BASED MOTIVATION LABEL
def infer_motivation(crime):
    crime = crime.lower()
    if "robbery" in crime or "theft" in crime:
        return "financial"
    elif "intimate partner" in crime or "rape" in crime:
        return "emotional"
    elif "assault" in crime or "weapon" in crime:
        return "power"
    else:
        return "unknown"

df["initial_motivation"] = df["crm_cd_desc"].apply(infer_motivation)


In [15]:

print(df["crime_text"][1])

On 2020-12-02 at 22 hours, in 77th street area, a 21.0-year-old M was involved in robbery at street. Weapon used: verbal threat. Case status: invest cont.


In [16]:
# Save processed data as CSV
df.to_csv("processed_crime_data.csv", index=False)

# -------------------------------
# Prepare JSON data for LLM usage
# -------------------------------

llm_data = []

for _, row in df.iterrows():
    llm_data.append({
        "text": row["crime_text"],
        "area": row["area_name"],
        "crime_type": row["crm_cd_desc"],
        "weapon": row["weapon_desc"],
        "motivation_hint": row["initial_motivation"]
    })

# Save LLM-ready JSON file
import json

with open("crime_data_llm_ready.json", "w", encoding="utf-8") as f:
    json.dump(llm_data, f, indent=2, ensure_ascii=False)


Training samples: 81736
initial_motivation
power        50029
emotional    19450
financial    12257
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ LLM-1 PERFORMANCE

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

   emotional       1.00      1.00      1.00      3890
   financial       1.00      1.00      1.00      2452
       power       1.00      1.00      1.00     10006

    accuracy                           1.00     16348
   macro avg       1.00      1.00      1.00     16348
weighted avg       1.00      1.00      1.00     16348


✅ LLM-1 model saved as /content/llm1_model.pkl


In [51]:
# =====================================================
# LLM-1 SUPERVISED TRAINING (MOTIVATION CLASSIFIER)
# =====================================================
# FIX 1: crm_cd_desc removed from crime_text (done in Cell 11)
# FIX 2: GroupShuffleSplit groups by crime type →
#         test set contains crime types NEVER seen in training.
#         A random split would let the model memorise per-crime-type patterns.
# =====================================================

import pandas as pd
import joblib
import json
import os
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
from sklearn.metrics import classification_report, accuracy_score

# ── STEP 1: Load CSV ────────────────────────────────
df = pd.read_csv("/content/processed_crime_data.csv")

# Keep required columns — include crm_cd_desc for grouping ONLY (not in text)
df = df[["crime_text", "initial_motivation", "crm_cd_desc"]]
df = df[df["initial_motivation"] != "unknown"].dropna().reset_index(drop=True)

print("Training samples:", len(df))
print(df["initial_motivation"].value_counts())
print("\nUnique crime types:", df["crm_cd_desc"].nunique())

# ── STEP 2: Encode Crime Text ────────────────────────
# Encode BEFORE split — embeddings are label-independent, no leakage here
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X = embedder.encode(
    df["crime_text"].tolist(),
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)
y = df["initial_motivation"].tolist()
groups = df["crm_cd_desc"].tolist()  # used for GroupShuffleSplit only

# ── STEP 3: GroupShuffleSplit — Unseen Crime Types ───
# This ensures the TEST SET contains crime types the model NEVER trained on.
# If accuracy is still 100% after this, there is another issue to investigate.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train = [y[i] for i in train_idx]
y_test  = [y[i] for i in test_idx]

train_types = set(groups[i] for i in train_idx)
test_types  = set(groups[i] for i in test_idx)
overlap     = train_types & test_types

print(f"\nTrain crime types : {len(train_types)}")
print(f"Test crime types  : {len(test_types)}")
print(f"Overlapping types : {len(overlap)}  ← should be 0 or very few")
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

# ── STEP 4: Train Classifier ─────────────────────────
clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
clf.fit(X_train, y_train)

# ── STEP 5: Evaluate ─────────────────────────────────
y_pred = clf.predict(X_test)
accuracy = round(accuracy_score(y_test, y_pred), 3)

# Cross-val on training data only (do not touch test set)
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="f1_weighted")
cv_mean_f1 = round(float(np.mean(cv_scores)), 3)

print("\n✅ LLM-1 PERFORMANCE (Real Generalization)\n")
print(f"Accuracy on UNSEEN crime types : {accuracy}")
print(f"CV Mean F1 (train set, 5-fold) : {cv_mean_f1}")
print("\nNOTE: 70-85% here is healthy and expected.")
print("      100% would mean the bias is still present somewhere.\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# ── STEP 6: Save Model ────────────────────────────────
SAVE_DIR = "/content/llm1_model"
os.makedirs(SAVE_DIR, exist_ok=True)

embedder.save(f"{SAVE_DIR}/embedder")
joblib.dump(clf, f"{SAVE_DIR}/classifier.joblib")

metadata = {
    "labels": clf.classes_.tolist(),
    "accuracy": accuracy,
    "cv_mean_f1": cv_mean_f1,
    "split_method": "GroupShuffleSplit_by_crime_type",
    "note": "Accuracy measured on crime types NOT seen during training"
}
with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ LLM-1 model saved to {SAVE_DIR}/")
print(f"   └── embedder/         (SentenceTransformer)")
print(f"   └── classifier.joblib (LogisticRegression)")
print(f"   └── metadata.json     (accuracy on unseen crime types)")


Training samples: 278155
initial_motivation
power        171537
emotional     63329
financial     43289
Name: count, dtype: int64

Unique crime types: 37


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4347 [00:00<?, ?it/s]


Train crime types : 27
Test crime types  : 10
Overlapping types : 0  ← should be 0 or very few
Train size: 202102 | Test size: 76053

✅ LLM-1 PERFORMANCE (Real Generalization)

Accuracy on UNSEEN crime types : 0.222
CV Mean F1 (train set, 5-fold) : 1.0

NOTE: 70-85% here is healthy and expected.
      100% would mean the bias is still present somewhere.

Classification Report:
              precision    recall  f1-score   support

   emotional       1.00      0.00      0.00     59256
   financial       0.97      1.00      0.98      2271
       power       0.20      0.99      0.33     14526

    accuracy                           0.22     76053
   macro avg       0.72      0.67      0.44     76053
weighted avg       0.85      0.22      0.10     76053



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ LLM-1 model saved to /content/llm1_model/
   └── embedder/         (SentenceTransformer)
   └── classifier.joblib (LogisticRegression)
   └── metadata.json     (accuracy on unseen crime types)


In [17]:
# =====================================================
# LLM-1 SUPERVISED TRAINING (MOTIVATION CLASSIFIER)
# =====================================================

import pandas as pd
import joblib
import json
import os

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# =====================================================
# STEP 1: Load CSV
# =====================================================
df = pd.read_csv("/content/processed_crime_data.csv")

# Keep only required columns
df = df[["crime_text", "initial_motivation"]]

# Remove unknown motivations
df = df[df["initial_motivation"] != "unknown"]

# Drop missing values
df = df.dropna()

print("Training samples:", len(df))
print(df["initial_motivation"].value_counts())

# =====================================================
# STEP 2: Encode Crime Text
# =====================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X = embedder.encode(
    df["crime_text"].tolist(),
    convert_to_numpy=True
)

y = df["initial_motivation"].tolist()

# =====================================================
# STEP 3: Train-Test Split
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================================
# STEP 4: Train Classifier
# =====================================================
clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

clf.fit(X_train, y_train)

# =====================================================
# STEP 5: Evaluate LLM-1
# =====================================================
y_pred = clf.predict(X_test)

accuracy = round(accuracy_score(y_test, y_pred), 3)

# Cross-validation F1
cv_scores = cross_val_score(clf, X, y, cv=5, scoring="f1_weighted")
cv_mean_f1 = round(float(np.mean(cv_scores)), 3)

print("\n✅ LLM-1 PERFORMANCE\n")
print("Accuracy:", accuracy)
print("CV Mean F1:", cv_mean_f1)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# =====================================================
# STEP 6: Save Trained Model (version-proof)
# =====================================================
SAVE_DIR = "/content/llm1_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save embedder (SentenceTransformer native format — no pickle)
embedder.save(f"{SAVE_DIR}/embedder")

# Save classifier (joblib — safe for sklearn, version-stable)
joblib.dump(clf, f"{SAVE_DIR}/classifier.joblib")

# Save metadata
metadata = {
    "labels": clf.classes_.tolist(),
    "accuracy": accuracy,
    "cv_mean_f1": cv_mean_f1
}
with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n✅ LLM-1 model saved to /content/llm1_model/")
print(f"   └── embedder/          (SentenceTransformer)")
print(f"   └── classifier.joblib  (LogisticRegression)")
print(f"   └── metadata.json      (labels, accuracy, cv_f1)")

Training samples: 278155
initial_motivation
power        171537
emotional     63329
financial     43289
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ LLM-1 PERFORMANCE

Accuracy: 1.0
CV Mean F1: 1.0

Classification Report:

              precision    recall  f1-score   support

   emotional       1.00      1.00      1.00     12666
   financial       1.00      1.00      1.00      8658
       power       1.00      1.00      1.00     34307

    accuracy                           1.00     55631
   macro avg       1.00      1.00      1.00     55631
weighted avg       1.00      1.00      1.00     55631



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ LLM-1 model saved to /content/llm1_model/
   └── embedder/          (SentenceTransformer)
   └── classifier.joblib  (LogisticRegression)
   └── metadata.json      (labels, accuracy, cv_f1)


## LLM2


In [18]:
import pandas as pd
import json

# ===============================
# STEP 1: Load Murder Motives Data
# ===============================
df = pd.read_csv("/content/Murder Motives.csv")

# ===============================
# STEP 2: Clean Column Names
# ===============================
df.columns = (
    df.columns.str.lower()
              .str.replace(" ", "_")
              .str.replace("/", "_")
)

# ===============================
# STEP 3: Verify Column Names
# ===============================
# Check if the necessary column exists
required_columns = [
    "gain",
    "property_dispute",
    "personal_vendetta_or_enemity",
    "love_affairs_sexual_relations",
    "dowry",
    "communalism",
    "casteism",
    "political_reasons",
    "terrorists_extremists",
    "other_causes"
]

# Print the actual column names in the DataFrame to debug
print("Actual column names in the DataFrame:", df.columns)

# Check if the required columns exist in the DataFrame
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"Warning: The following columns are missing in the dataset: {missing_cols}")
else:
    print("All required columns are present.")

# ===============================
# STEP 4: Remove Aggregate Rows
# ===============================
df = df[~df["state"].str.contains("TOTAL", na=False)]

# ===============================
# STEP 5: Identify Motivation Columns (LLM-2)
# ===============================
# Ensure the column names match with the cleaned names
motivation_cols = [
    "gain",
    "property_dispute",
    "personal_vendetta_or_enemity",
    "love_affairs_sexual_relations",  # This should match the column name exactly
    "dowry",
    "communalism",
    "casteism",
    "political_reasons",
    "terrorists_extremists",
    "other_causes"
]

# ===============================
# STEP 6: Convert Each Row to Motivation Profile
# ===============================
llm2_data = []

for _, row in df.iterrows():
    # Build motivation profile for each row
    motivation_profile = {
        col: int(row[col]) if not pd.isna(row[col]) else 0
        for col in motivation_cols if col in df.columns  # Ensure the column exists
    }

    llm2_data.append({
        "state": row["state"],
        "year": row["year"],
        "motivation_distribution": motivation_profile
    })

# ===============================
# STEP 7: Save LLM-2 Dataset
# ===============================
with open("llm2_motivation_dataset.json", "w") as f:
    json.dump(llm2_data, f, indent=2)

print("✅ LLM-2 Motivation Dataset Created")


Actual column names in the DataFrame: Index(['state', 'year', 'gain', 'property_dispute',
       'personal_vendetta_or_enemity', 'love_affairs__sexual_relations',
       'dowry', 'lunacy', 'witchcraft', 'communalism', 'casteism',
       'class_conflict', 'political_reasons', 'terrorists__extremists',
       'other_causes', 'total'],
      dtype='object')
✅ LLM-2 Motivation Dataset Created


In [ ]:
import json
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

# =====================================================
# STEP 1 — LOAD JSON
# =====================================================
with open("/content/llm2_motivation_dataset.json", "r") as f:
    data = json.load(f)

# =====================================================
# STEP 2 — FLATTEN DATA
# =====================================================
records = []

for entry in data:
    record = {
        "state": entry["state"],
        "year": entry["year"]
    }
    record.update(entry["motivation_distribution"])
    records.append(record)

df = pd.DataFrame(records)

# =====================================================
# STEP 3 — ENCODE STATE
# =====================================================
le = LabelEncoder()
df["state_encoded"] = le.fit_transform(df["state"])

# =====================================================
# STEP 4 — FEATURES & TARGETS
# =====================================================
X = df[["state_encoded", "year"]]

motivation_cols = [
    col for col in df.columns
    if col not in ["state", "year", "state_encoded"]
]

y = df[motivation_cols]

# =====================================================
# STEP 5 — TRAIN-TEST SPLIT
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =====================================================
# STEP 6 — TRAIN LLM-2
# =====================================================
model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )
)

model.fit(X_train, y_train)

# =====================================================
# STEP 7 — EVALUATION
# =====================================================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"\n✅ LLM-2 Mean Absolute Error: {round(mae, 2)}")

# =====================================================
# STEP 8 — SAVE MODEL
# =====================================================
with open("/content/llm2_model.pkl", "wb") as f:
    pickle.dump(
        {
            "model": model,
            "state_encoder": le,
            "motivation_columns": motivation_cols
        },
        f
    )

print("✅ LLM-2 model saved at /content/llm2_model.pkl")


In [19]:
import json
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

# =====================================================
# STEP 1 — LOAD JSON
# =====================================================
with open("/content/llm2_motivation_dataset.json", "r") as f:
    data = json.load(f)

# =====================================================
# STEP 2 — FLATTEN DATA
# =====================================================
records = []

for entry in data:
    record = {
        "state": entry["state"],
        "year": entry["year"]
    }
    record.update(entry["motivation_distribution"])
    records.append(record)

df = pd.DataFrame(records)

# =====================================================
# STEP 3 — ENCODE STATE
# =====================================================
le = LabelEncoder()
df["state_encoded"] = le.fit_transform(df["state"])

# =====================================================
# STEP 4 — FEATURES & TARGETS
# =====================================================
X = df[["state_encoded", "year"]]

motivation_cols = [
    col for col in df.columns
    if col not in ["state", "year", "state_encoded"]
]

y = df[motivation_cols]

# =====================================================
# STEP 5 — TRAIN-TEST SPLIT
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =====================================================
# STEP 6 — TRAIN LLM-2
# =====================================================
model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )
)

model.fit(X_train, y_train)

# =====================================================
# STEP 7 — EVALUATION
# =====================================================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"\n✅ LLM-2 Mean Absolute Error: {round(mae, 2)}")

# =====================================================
# STEP 8 — SAVE MODEL (version-proof)
# =====================================================
import os, json

SAVE_DIR = "/content/llm2_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model + encoder together (both are sklearn — joblib is safe)
joblib.dump(model, f"{SAVE_DIR}/model.joblib")
joblib.dump(le,    f"{SAVE_DIR}/state_encoder.joblib")

# Save metadata as plain JSON
with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump({
        "motivation_columns": motivation_cols,
        "mae": round(mae, 2)
    }, f, indent=2)

print("✅ LLM-2 model saved to /content/llm2_model/")
print(f"   └── model.joblib")
print(f"   └── state_encoder.joblib")
print(f"   └── metadata.json")


✅ LLM-2 Mean Absolute Error: 40.98
✅ LLM-2 model saved to /content/llm2_model/
   └── model.joblib
   └── state_encoder.joblib
   └── metadata.json


In [20]:
# =========================
# LLM-2 METRICS
# =========================

def run_llm2_with_metrics(user_input):
    print("\n==============================")
    print("🚀 RUNNING LLM-2 WITH METRICS")
    print("==============================\n")

    response, latency = _measure_latency(llm2, user_input)

    metrics = {
        "latency_ms": latency,
        "response_length": len(response),
        "confidence_score": _confidence_score(llm2, response),
        "quality_score": _quality_score(llm2, response, user_input),
    }

    response_2 = llm2(user_input)
    metrics["consistency_score"] = _consistency_score(response, response_2)

    metrics["final_score"] = _final_score(metrics)

    print("📤 LLM-2 OUTPUT:\n")
    print(response)

    print("\n📊 LLM-2 METRICS:\n")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    print(f"\n🏁 LLM-2 FINAL SCORE: {metrics['final_score']}/100")
    print("\n==============================\n")

    return response, metrics


## llm 3

In [21]:
import pandas as pd
import json

# ===============================
# STEP 1: Load Chicago Crimes Data
# ===============================
df = pd.read_csv("/content/Chicago_Crimes_2022.csv")

# ===============================
# STEP 2: Standardize Column Names
# ===============================
df.columns = (
    df.columns.str.lower()
              .str.replace(" ", "_")
              .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# ===============================
# STEP 3: Select Context-Relevant Columns (LLM-3)
# ===============================
llm3_cols = [
    "date",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "beat",
    "district",
    "ward",
    "community_area",
    "year",
    "latitude",
    "longitude"
]

df_llm3 = df[llm3_cols].fillna("UNKNOWN")

# ===============================
# STEP 4: Create Context Narrative (Text for LLM)
# ===============================
def build_context_text(row):
    return (
        f"In {row['year']}, a {row['primary_type']} incident occurred at a "
        f"{row['location_description']} location. "
        f"Domestic case: {row['domestic']}. "
        f"Arrest made: {row['arrest']}. "
        f"District {row['district']}, Beat {row['beat']}."
    )

df_llm3["context_text"] = df_llm3.apply(build_context_text, axis=1)

# ===============================
# STEP 5: Convert to LLM-3 JSON Format
# ===============================
llm3_data = []

for _, row in df_llm3.iterrows():
    llm3_data.append({
        "context_text": row["context_text"],
        "crime_type": row["primary_type"],
        "location_type": row["location_description"],
        "domestic": row["domestic"],
        "arrest": row["arrest"],
        "district": row["district"],
        "year": row["year"]
    })

# ===============================
# STEP 6: Save LLM-3 Dataset
# ===============================
with open("llm3_background_context.json", "w") as f:
    json.dump(llm3_data, f, indent=2)

print("✅ LLM-3 Background & Context Dataset Created")

/tmp/ipython-input-1543/2375959944.py:7: DtypeWarning: Columns (0,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/Chicago_Crimes_2022.csv")


✅ LLM-3 Background & Context Dataset Created


In [23]:
import json
import joblib
import numpy as np
import os

from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

# =====================================================
# STEP 1 — LOAD LLM-3 DATA
# =====================================================
with open("/content/llm3_background_context.json", "r") as f:
    data = json.load(f)

texts = [item["context_text"] for item in data]

# =====================================================
# STEP 2 — ENCODE CONTEXT TEXT
# =====================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X = embedder.encode(
    texts,
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)

# =====================================================
# STEP 3 — FAST CLUSTERING (NO SILHOUETTE)
# =====================================================
NUM_CLUSTERS = 5

kmeans = MiniBatchKMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=42,
    batch_size=256
)

cluster_labels = kmeans.fit_predict(X)

# =====================================================
# STEP 4 — ATTACH CLUSTERS
# =====================================================
for i, item in enumerate(data):
    item["llm3_cluster"] = int(cluster_labels[i])

# =====================================================
# STEP 5 — SAVE MODEL & DATA (version-proof)
# =====================================================
SAVE_DIR = "/content/llm3_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save embedder (SentenceTransformer native format — no pickle)
embedder.save(f"{SAVE_DIR}/embedder")

# Save kmeans (sklearn — joblib is safe)
joblib.dump(kmeans, f"{SAVE_DIR}/kmeans.joblib")

# Save metadata
with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump({
        "num_clusters": NUM_CLUSTERS,
        "total_records": len(data)
    }, f, indent=2)

# Save clustered data as JSON (already safe)
with open("/content/llm3_clustered_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("✅ LLM-3 training complete")
print(f"   └── embedder/        (SentenceTransformer)")
print(f"   └── kmeans.joblib    (MiniBatchKMeans)")
print(f"   └── metadata.json    (num_clusters, total_records)")
print(f"   llm3_clustered_data.json saved to /content/")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3084 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ LLM-3 training complete
   └── embedder/        (SentenceTransformer)
   └── kmeans.joblib    (MiniBatchKMeans)
   └── metadata.json    (num_clusters, total_records)
   llm3_clustered_data.json saved to /content/


In [24]:
# =========================
# LLM-3 METRICS
# =========================

def run_llm3_with_metrics(user_input):
    print("\n==============================")
    print("🚀 RUNNING LLM-3 WITH METRICS")
    print("==============================\n")

    response, latency = _measure_latency(llm3, user_input)

    metrics = {
        "latency_ms": latency,
        "response_length": len(response),
        "confidence_score": _confidence_score(llm3, response),
        "quality_score": _quality_score(llm3, response, user_input),
    }

    response_2 = llm3(user_input)
    metrics["consistency_score"] = _consistency_score(response, response_2)

    metrics["final_score"] = _final_score(metrics)

    print("📤 LLM-3 OUTPUT:\n")
    print(response)

    print("\n📊 LLM-3 METRICS:\n")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    print(f"\n🏁 LLM-3 FINAL SCORE: {metrics['final_score']}/100")
    print("\n==============================\n")

    return response, metrics


In [ ]:
out3, m3 = run_llm3_with_metrics(prompt)


# llm4

In [25]:
# STEP 1 — LOAD ALL PRE-TRAINED MODELS

import joblib
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# ── LLM-1 ──────────────────────────────────────────
llm1_embedder   = SentenceTransformer("/content/llm1_model/embedder")
llm1_classifier = joblib.load("/content/llm1_model/classifier.joblib")
with open("/content/llm1_model/metadata.json") as f:
    llm1_metadata = json.load(f)

print(f"✅ LLM-1 loaded | Accuracy: {llm1_metadata['accuracy']} | Classes: {llm1_metadata['labels']}")

# ── LLM-2 ──────────────────────────────────────────
llm2_model         = joblib.load("/content/llm2_model/model.joblib")
llm2_state_encoder = joblib.load("/content/llm2_model/state_encoder.joblib")
with open("/content/llm2_model/metadata.json") as f:
    llm2_metadata = json.load(f)

print(f"✅ LLM-2 loaded | MAE: {llm2_metadata['mae']}")

# ── LLM-3 ──────────────────────────────────────────
llm3_embedder = SentenceTransformer("/content/llm3_model/embedder")
llm3_kmeans   = joblib.load("/content/llm3_model/kmeans.joblib")
with open("/content/llm3_model/metadata.json") as f:
    llm3_metadata = json.load(f)

print(f"✅ LLM-3 loaded | {llm3_kmeans.n_clusters} clusters")

# ── DATASETS ───────────────────────────────────────
with open("/content/llm2_motivation_dataset.json", "r", encoding="utf-8") as f:
    llm2_dataset = json.load(f)

with open("/content/llm3_clustered_data.json", "r", encoding="utf-8") as f:
    llm3_dataset = json.load(f)

print(f"✅ LLM-2 dataset | {len(llm2_dataset)} records")
print(f"✅ LLM-3 dataset | {len(llm3_dataset)} records")

print("\n✅ All models loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM-1 loaded | Accuracy: 1.0 | Classes: ['emotional', 'financial', 'power']
✅ LLM-2 loaded | MAE: 40.98


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM-3 loaded | 5 clusters
✅ LLM-2 dataset | 458 records
✅ LLM-3 dataset | 197313 records

✅ All models loaded successfully!


#test

In [26]:
# Install required packages
!pip install google-generativeai sentence-transformers scikit-learn pandas numpy -q
print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [52]:
from google.colab import userdata
import os

print("🔑 Setting up API keys for all 4 LLM stages...\n")

# Initialize keys
llm1_api_key = None
llm2_api_key = None
llm3_api_key = None
llm4_api_key = None

try:
    # Try to get from Colab Secrets (recommended)
    # These should be the actual names of the secrets you created in Colab
    try:
        llm1_api_key = userdata.get('LLM1_API_KEY')
    except userdata.SecretNotFoundError:
        pass # LLM1 is optional, if not found, it's fine

    llm2_api_key = userdata.get('LLM2_API_KEY') # This one is required

    try:
        llm3_api_key = userdata.get('LLM3_API_KEY')
    except userdata.SecretNotFoundError:
        pass # LLM3 is optional, if not found, it's fine

    llm4_api_key = userdata.get('LLM4_API_KEY') # This one is required

    if llm2_api_key and llm4_api_key:
        os.environ['LLM1_API_KEY'] = llm1_api_key if llm1_api_key else "PLACEHOLDER"
        os.environ['LLM2_API_KEY'] = llm2_api_key
        os.environ['LLM3_API_KEY'] = llm3_api_key if llm3_api_key else "PLACEHOLDER"
        os.environ['LLM4_API_KEY'] = llm4_api_key
        print("✅ API keys loaded from Colab Secrets")
        print(f"  LLM-1: {'✅ Configured' if llm1_api_key else '⚠️  Not needed (local model)'}")
        print(f"  LLM-2: ✅ Configured")
        print(f"  LLM-3: {'✅ Configured' if llm3_api_key else '⚠️  Not needed (local model)'}")
        print(f"  LLM-4: ✅ Configured")
    else:
        # If required keys are missing, raise a specific error
        raise ValueError("❌ Required API keys (LLM2_API_KEY, LLM4_API_KEY) not found in Colab Secrets.")

except userdata.NotebookAccessError:
    print("❌ Could not access Colab Secrets")
    print("Please enable notebook access for secrets.")
    raise

except ValueError as e:
    print(e)
    print("\n📌 Fix this in Colab:")
    print("  1. Click 🔑 Secrets (left sidebar)")
    print("  2. Add:")
    print("     - LLM2_API_KEY (required)")
    print("     - LLM4_API_KEY (required)")
    print("     - LLM1_API_KEY (optional)")
    print("     - LLM3_API_KEY (optional)")
    print("  3. Re-run this cell")
    raise

🔑 Setting up API keys for all 4 LLM stages...

✅ API keys loaded from Colab Secrets
  LLM-1: ✅ Configured
  LLM-2: ✅ Configured
  LLM-3: ✅ Configured
  LLM-4: ✅ Configured


In [53]:
def extract_json_from_response(text: str) -> dict:
    """Robustly extract JSON from Gemini response (handles markdown wrappers)."""
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass

    # Try extracting from markdown code blocks
    json_pattern = r'```json\s*({.*?})\s*```'
    match = re.search(json_pattern, text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # Try extracting any JSON object
    json_pattern = r'\{.*\}'
    match = re.search(json_pattern, text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    raise ValueError(f"Could not extract valid JSON from response: {text[:200]}...")


def call_gemini_with_retry(prompt_text: str, system_prompt: str, api_key: str = None, max_retries: int = 3) -> str:
    """Call Gemini API with retry logic and proper format."""
    # Use provided key or default to LLM2 key
    key = api_key or GEMINI_API_KEY_LLM2

    for attempt in range(max_retries):
        try:
            genai.configure(api_key=key)  # Configure with specific key
            model = genai.GenerativeModel(
                model_name="gemini-2.5-flash",
                system_instruction=system_prompt
            )
            response = model.generate_content(prompt_text)
            return response.text
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"⚠️ Gemini API error (attempt {attempt+1}/{max_retries}): {str(e)[:100]}")
                time.sleep(2 ** attempt)
            else:
                raise

    raise RuntimeError("Failed to get response from Gemini after retries")


In [54]:
import google.generativeai as genai
import json
import re
import time
import pandas as pd
import pickle
import numpy as np

GEMINI_API_KEY_LLM1 = os.environ.get('LLM1_API_KEY')
GEMINI_API_KEY_LLM2 = os.environ.get('LLM2_API_KEY')
GEMINI_API_KEY_LLM3 = os.environ.get('LLM3_API_KEY')
GEMINI_API_KEY_LLM4 = os.environ.get('LLM4_API_KEY')

if not all([GEMINI_API_KEY_LLM1, GEMINI_API_KEY_LLM2, GEMINI_API_KEY_LLM3, GEMINI_API_KEY_LLM4]):
    raise ValueError("❌ All 4 API keys required! (LLM1_API_KEY, LLM2_API_KEY, LLM3_API_KEY, LLM4_API_KEY)")

# Configure Gemini with LLM-1 key initially
genai.configure(api_key=GEMINI_API_KEY_LLM1)
print("✅ Gemini API configured for all 4 LLM stages")
print(f"   LLM-1: Using dedicated API key (LLM1_API_KEY)")
print(f"   LLM-2: Using dedicated API key (LLM2_API_KEY)")
print(f"   LLM-3: Using dedicated API key (LLM3_API_KEY)")
print(f"   LLM-4: Using dedicated API key (LLM4_API_KEY)")

# Create output directory
OUTPUT_DIR = '/content/analysis_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR}")

✅ Gemini API configured for all 4 LLM stages
   LLM-1: Using dedicated API key (LLM1_API_KEY)
   LLM-2: Using dedicated API key (LLM2_API_KEY)
   LLM-3: Using dedicated API key (LLM3_API_KEY)
   LLM-4: Using dedicated API key (LLM4_API_KEY)
📁 Output directory: /content/analysis_results/


In [55]:
LLM1_SYSTEM_PROMPT = """
You are an expert criminal psychologist and crime motivation analyst.

Your task:
1. Analyze the crime description carefully
2. Identify the PRIMARY motivation behind the crime
3. Provide reasoning based ONLY on facts presented
4. Assign a confidence level reflecting your certainty

Possible motivations:
- emotional: Personal conflicts, domestic disputes, revenge, rage, jealousy
- financial: Theft, robbery, fraud, extortion, monetary gain
- power: Assault, murder, control, dominance, territorial disputes
- sexual: Sexual assault, exploitation, predatory behavior
- unknown: Insufficient information to determine

CHAIN-OF-THOUGHT REASONING (CRITICAL):
Before reaching your conclusion, work through these steps:
1. EXTRACT KEY FACTS: Identify concrete details from the crime description
2. IDENTIFY INDICATORS: List specific behavioral or contextual indicators
3. MAP TO MOTIVATIONS: Connect each indicator to possible motivations
4. EVALUATE EVIDENCE: Rate the strength of evidence for each motivation
5. RESOLVE CONFLICTS: If multiple motivations appear, determine which is primary
6. ASSESS CONFIDENCE: Consider evidence quality and certainty level

IMPORTANT RULES:
1. Respond ONLY with valid JSON (no markdown, no explanations outside JSON)
2. Never assume missing information
3. Base reasoning on explicit crime details
4. Return exactly these JSON keys: predicted_motivation, confidence, reasoning, crime_indicators, reasoning_chain
5. Confidence must be one of: Low, Medium, High
6. reasoning_chain MUST be an array of strings (one step per element)
7. Each step in reasoning_chain must clearly explain that reasoning step
"""

LLM2_SYSTEM_PROMPT = """
You are a forensic crime analyst specializing in historical motivation patterns.

Your task:
1. Analyze historical motivation distributions
2. Identify dominant motivation patterns for the crime type
3. Correlate patterns with case context (state, year, crime type)
4. Provide confidence level for your assessment

CHAIN-OF-THOUGHT REASONING (CRITICAL):
Before reaching your conclusion, work through these steps:
1. EXTRACT CONTEXT: What state, year, and crime type are relevant?
2. QUERY HISTORICAL DATA: What patterns exist for this combination?
3. IDENTIFY DISTRIBUTION: Which motivations are most common?
4. ASSESS DATA QUALITY: How exact is your historical data?
5. HANDLE MISSING DATA: If data is missing, how do you extrapolate?
6. DETERMINE CONFIDENCE: Based on data quality, how confident are you?
7. RANK ALTERNATIVES: What are secondary/tertiary motivations?

IMPORTANT RULES:
1. Respond ONLY with valid JSON (no markdown, no explanations)
2. Never invent statistics or numbers
3. Use provided historical data as primary source
4. Handle missing data gracefully
5. Return JSON keys: dominant_historical_motivation, confidence_level, explanation, data_quality, reasoning_chain
6. Confidence must be: Low, Medium, High
7. data_quality must be: Exact, Partial, Estimated
8. reasoning_chain MUST be an array of strings (one step per element)
"""

LLM3_SYSTEM_PROMPT = """
You are a behavioral pattern analyst for crime classification.

Your task:
1. Identify the crime context pattern based on structured information
2. Classify the behavioral or situational pattern
3. Assess pattern confidence level
4. Do NOT infer motivation (that is handled by LLM-1)

Possible patterns:
- domestic_violence_pattern: Personal/family relationships, intimate partners
- public_street_crime_pattern: Stranger crimes, public locations
- weapon_escalation_pattern: Weapon involvement, armed crimes
- opportunistic_crime_pattern: Casual, minimal planning
- organized_repeat_crime_pattern: Organized groups, repeat offenders
- unclear_general_pattern: Insufficient information

CHAIN-OF-THOUGHT REASONING (CRITICAL):
Before reaching your conclusion, work through these steps:
1. PARSE CONTEXT: What key details are provided in the crime context?
2. IDENTIFY BEHAVIORAL MARKERS: What patterns of behavior are evident?
3. CLASSIFY PATTERN TYPE: Which of the possible patterns best fits?
4. CROSS-CHECK INDICATORS: What specific indicators support this pattern?
5. ELIMINATE ALTERNATIVES: Why other patterns don't fit as well?
6. ASSESS CERTAINTY: How confident are you in this classification?
7. NOTE EDGE CASES: Are there ambiguous elements?

IMPORTANT RULES:
1. Respond ONLY with valid JSON (no markdown, no explanations)
2. Never assume information not provided
3. Return JSON keys: identified_pattern, confidence_level, explanation, pattern_indicators, reasoning_chain
4. Confidence must be: Low, Medium, High
5. pattern_indicators should be array of observed behavioral markers
6. reasoning_chain MUST be an array of strings (one step per element)
"""

LLM4_SYSTEM_PROMPT = """
You are an expert forensic psychologist and crime analysis report writer.

Your task:
1. Generate a comprehensive, structured analytical report
2. Use ONLY the provided fused model output and supporting analysis
3. Do NOT introduce new facts or assumptions beyond what is provided
4. Explain the reasoning chain, confidence level, and limitations
5. Provide clear actionable insights

CHAIN-OF-THOUGHT REASONING (CRITICAL):
Before writing your report, reason through these steps:
1. SYNTHESIZE INPUTS: What did each of the 3 LLMs conclude?
2. ASSESS AGREEMENT: Which models agree? Which disagree?
3. WEIGHT CONTRIBUTIONS: How strongly did each model influence the final decision?
4. IDENTIFY CONFIDENCE DRIVERS: What makes us confident or uncertain?
5. DETECT CONFLICTS: Are there contradictions to reconcile?
6. BUILD NARRATIVE: How do you explain the decision flow?
7. DETERMINE RECOMMENDATIONS: What should happen next?

REPORT STRUCTURE:
1. Executive Summary: Main conclusion in 1-2 sentences
2. Confidence Assessment: Explain confidence level and what it means
3. Reasoning Chain: Step-by-step how the conclusion was reached
4. Model Contributions: How each model influenced the decision
5. Supporting Evidence: Key indicators from the analysis
6. Conflict Analysis: If models disagreed, explain the disagreement
7. Recommendations: Next steps based on confidence and findings
8. Limitations: What additional information would help

IMPORTANT RULES:
1. Be objective and evidence-based
2. Clearly distinguish between high/medium/low confidence
3. Never make statements beyond the data provided
4. Explicitly state your reasoning chain in the report
"""

print("✅ System prompts defined WITH CHAIN-OF-THOUGHT")

✅ System prompts defined WITH CHAIN-OF-THOUGHT


In [56]:
def load_models():
    """Load all pre-trained models and datasets."""

    print("\n📦 Loading pre-trained models...\n")

    import torch
    import joblib, json
    from sentence_transformers import SentenceTransformer

    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
        device_name = "🔴 GPU (CUDA)"
    else:
        device = torch.device('cpu')
        device_name = "🟢 CPU"

    print(f"   Device: {device_name}")
    print(f"   Using: {device}\n")

    try:
        # ── LLM-1 ──────────────────────────────────────────
        llm1_embedder   = SentenceTransformer("/content/llm1_model/embedder")
        llm1_classifier = joblib.load("/content/llm1_model/classifier.joblib")
        with open("/content/llm1_model/metadata.json") as f:
            llm1_metadata = json.load(f)
        print(f"✅ LLM-1 loaded | Accuracy: {llm1_metadata['accuracy']}")

        # ── LLM-2 ──────────────────────────────────────────
        llm2_model         = joblib.load("/content/llm2_model/model.joblib")
        llm2_state_encoder = joblib.load("/content/llm2_model/state_encoder.joblib")
        with open("/content/llm2_model/metadata.json") as f:
            llm2_metadata = json.load(f)
        print(f"✅ LLM-2 loaded | MAE: {llm2_metadata['mae']}")

        # ── LLM-3 ──────────────────────────────────────────
        llm3_embedder = SentenceTransformer("/content/llm3_model/embedder")
        llm3_kmeans   = joblib.load("/content/llm3_model/kmeans.joblib")
        with open("/content/llm3_model/metadata.json") as f:
            llm3_metadata = json.load(f)
        print(f"✅ LLM-3 loaded | {llm3_kmeans.n_clusters} clusters")

        # ── DATASETS ───────────────────────────────────────
        with open("/content/llm2_motivation_dataset.json", "r", encoding="utf-8") as f:
            llm2_dataset = json.load(f)
        print(f"✅ LLM-2 dataset | {len(llm2_dataset)} historical records")

        with open("/content/llm3_clustered_data.json", "r", encoding="utf-8") as f:
            llm3_dataset = json.load(f)
        print(f"✅ LLM-3 dataset | {len(llm3_dataset)} context records")

        return {
            "llm1_embedder":    llm1_embedder,
            "llm1_classifier":  llm1_classifier,
            "llm1_metadata":    llm1_metadata,
            "llm2_model":       llm2_model,
            "llm2_state_encoder": llm2_state_encoder,
            "llm2_metadata":    llm2_metadata,
            "llm2_dataset":     llm2_dataset,
            "llm3_embedder":    llm3_embedder,
            "llm3_kmeans":      llm3_kmeans,
            "llm3_metadata":    llm3_metadata,
            "llm3_dataset":     llm3_dataset,
            "device":           device,
            "device_name":      device_name
        }

    except FileNotFoundError as e:
        print(f"❌ Error: Could not find model file: {e}")
        print("   Have you run all 3 training scripts with the new saving code?")
        raise

# Load models
MODELS = load_models()
print("\n✅ All models loaded successfully!")


📦 Loading pre-trained models...

   Device: 🔴 GPU (CUDA)
   Using: cuda



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM-1 loaded | Accuracy: 0.222
✅ LLM-2 loaded | MAE: 40.98


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM-3 loaded | 5 clusters
✅ LLM-2 dataset | 458 historical records
✅ LLM-3 dataset | 197313 context records

✅ All models loaded successfully!


In [57]:
def llm1_agent_from_preprocessed(record: dict) -> dict:
    """LLM-1: Analyze crime motivation using Gemini API."""
    try:
        user_prompt_text = f"""
Analyze this crime case step-by-step using the CHAIN-OF-THOUGHT approach:

📋 CRIME DETAILS:
Description: {record.get("crime_text", "Not available")}
Crime Type: {record.get("crm_cd_desc", "Unknown")}
Weapon: {record.get("weapon_desc", "Unknown")}
Location Type: {record.get("premis_desc", "Unknown")}
Victim Age: {record.get("vict_age", "Unknown")}
Victim Gender: {record.get("vict_sex", "Unknown")}
Area: {record.get("area_name", "Unknown")}
Status: {record.get("status_desc", "Unknown")}

FOLLOW THE CHAIN-OF-THOUGHT STEPS:
1. EXTRACT KEY FACTS: What concrete details stand out?
2. IDENTIFY INDICATORS: What behavioral/contextual clues do you see?
3. MAP TO MOTIVATIONS: Which motivations fit the indicators?
4. EVALUATE EVIDENCE: How strong is the evidence for each motivation?
5. RESOLVE CONFLICTS: Which motivation is PRIMARY?
6. ASSESS CONFIDENCE: How certain are you (Low/Medium/High)?

Include your complete reasoning_chain in the JSON response.
"""

        response_text = call_gemini_with_retry(user_prompt_text, LLM1_SYSTEM_PROMPT, api_key=GEMINI_API_KEY_LLM1)
        result = extract_json_from_response(response_text)

        return {
            "llm_stage": "LLM-1",
            "predicted_motivation": result.get("predicted_motivation", "unknown"),
            "confidence": result.get("confidence", "Low"),
            "reasoning": result.get("reasoning", "No reasoning provided"),
            "reasoning_chain": result.get("reasoning_chain", []),
            "crime_indicators": result.get("crime_indicators", []),
            "api_key_used": "LLM1_API_KEY"
        }

    except Exception as e:
        print(f"❌ LLM-1 failed: {e}")
        return {
            "llm_stage": "LLM-1",
            "predicted_motivation": "unknown",
            "confidence": "Low",
            "reasoning": f"Error: {str(e)[:100]}",
            "reasoning_chain": [],
            "crime_indicators": [],
            "api_key_used": "LLM1_API_KEY"
        }


def llm2_agent_from_preprocessed(context: dict, llm2_dataset: list) -> dict:
    """LLM-2: Analyze historical context using Gemini API."""
    try:
        matched_record = None

        if context.get("state") and context.get("year"):
            for record in llm2_dataset:
                if (record.get("state") == context.get("state") and
                   int(record.get("year", -1)) == int(context.get("year", -1))):
                    matched_record = record
                    break

        if matched_record:
            historical_text = json.dumps(matched_record["motivation_distribution"], indent=2)
            data_note = "Exact historical data found."
            data_quality = "Exact"
        else:
            historical_text = "Exact state/year data not available."
            data_note = "Using generalized historical trends."
            data_quality = "Estimated"

        user_prompt = f"""
Analyze historical motivation patterns step-by-step using CHAIN-OF-THOUGHT:

📊 CRIME CONTEXT:
State: {context.get("state", "Unknown")}
Year: {context.get("year", "Unknown")}
Crime Type: {context.get("crime_type", "Unknown")}

HISTORICAL DATA:
{historical_text}

Note: {data_note}

FOLLOW THE CHAIN-OF-THOUGHT STEPS:
1. EXTRACT CONTEXT: What state, year, and crime type are relevant?
2. QUERY HISTORICAL DATA: What patterns exist for this combination?
3. IDENTIFY DISTRIBUTION: Which motivations are most common?
4. ASSESS DATA QUALITY: How exact is your historical data?
5. HANDLE MISSING DATA: If data is missing, how do you extrapolate?
6. DETERMINE CONFIDENCE: Based on data quality, how confident are you?
7. RANK ALTERNATIVES: What are secondary/tertiary motivations?

Include your complete reasoning_chain in the JSON response.
"""

        response_text = call_gemini_with_retry(user_prompt, LLM2_SYSTEM_PROMPT, api_key=GEMINI_API_KEY_LLM2)
        result = extract_json_from_response(response_text)

        return {
            "llm_stage": "LLM-2",
            "dominant_historical_motivation": result.get("dominant_historical_motivation", "unknown"),
            "confidence_level": result.get("confidence_level", "Low"),
            "explanation": result.get("explanation", "No explanation provided"),
            "data_quality": result.get("data_quality", data_quality),
            "reasoning_chain": result.get("reasoning_chain", []),
            "api_key_used": "LLM2_API_KEY"
        }

    except Exception as e:
        print(f"❌ LLM-2 failed: {e}")
        return {
            "llm_stage": "LLM-2",
            "dominant_historical_motivation": "unknown",
            "confidence_level": "Low",
            "explanation": f"Error: {str(e)[:100]}",
            "data_quality": "Estimated",
            "reasoning_chain": [],
            "api_key_used": "LLM2_API_KEY"
        }


def llm3_agent_from_preprocessed(record: dict) -> dict:
    """LLM-3: Identify behavioral patterns using Gemini API."""
    try:
        user_prompt = f"""
Identify the crime pattern step-by-step using CHAIN-OF-THOUGHT:

📋 CRIME CONTEXT:
Crime Type: {record.get("crime_type", "Unknown")}
Location: {record.get("location_type", "Unknown")}
Domestic: {record.get("domestic", "Unknown")}
Arrest: {record.get("arrest", "Unknown")}
District: {record.get("district", "Unknown")}
Year: {record.get("year", "Unknown")}

SUMMARY: {record.get("context_text", "Not provided")}

FOLLOW THE CHAIN-OF-THOUGHT STEPS:
1. PARSE CONTEXT: What key details are provided in the crime context?
2. IDENTIFY BEHAVIORAL MARKERS: What patterns of behavior are evident?
3. CLASSIFY PATTERN TYPE: Which of the possible patterns best fits?
4. CROSS-CHECK INDICATORS: What specific indicators support this pattern?
5. ELIMINATE ALTERNATIVES: Why other patterns don't fit as well?
6. ASSESS CERTAINTY: How confident are you in this classification?
7. NOTE EDGE CASES: Are there ambiguous elements?

Include your complete reasoning_chain in the JSON response.
"""

        response_text = call_gemini_with_retry(user_prompt, LLM3_SYSTEM_PROMPT, api_key=GEMINI_API_KEY_LLM3)
        result = extract_json_from_response(response_text)

        return {
            "llm_stage": "LLM-3",
            "identified_pattern": result.get("identified_pattern", "unclear_general_pattern"),
            "confidence_level": result.get("confidence_level", "Low"),
            "explanation": result.get("explanation", "No explanation provided"),
            "pattern_indicators": result.get("pattern_indicators", []),
            "reasoning_chain": result.get("reasoning_chain", []),
            "api_key_used": "LLM3_API_KEY"
        }

    except Exception as e:
        print(f"❌ LLM-3 failed: {e}")
        return {
            "llm_stage": "LLM-3",
            "identified_pattern": "unclear_general_pattern",
            "confidence_level": "Low",
            "explanation": f"Error: {str(e)[:100]}",
            "pattern_indicators": [],
            "reasoning_chain": [],
            "api_key_used": "LLM3_API_KEY"
        }


def llm4_generate_report(fusion_output: dict, llm1_result: dict, llm2_result: dict, llm3_result: dict, crime_details: dict = None) -> dict:
    """LLM-4: Generate comprehensive report with COMPLETE reasoning synthesis from all 3 models."""
    try:
        # Format reasoning chains for readability
        llm1_reasoning = "\n".join([f"  • {step}" for step in llm1_result.get("reasoning_chain", [])])
        llm2_reasoning = "\n".join([f"  • {step}" for step in llm2_result.get("reasoning_chain", [])])
        llm3_reasoning = "\n".join([f"  • {step}" for step in llm3_result.get("reasoning_chain", [])])

        user_prompt = f"""
SYNTHESIZE COMPLETE REASONING FROM ALL 3 MODELS:

═══════════════════════════════════════════════════════════════════════

LLM-1 (MOTIVATION ANALYSIS):
Conclusion: {llm1_result.get("predicted_motivation", "unknown")} | Confidence: {llm1_result.get("confidence", "Low")}
Reasoning Chain:
{llm1_reasoning if llm1_reasoning.strip() else "  • No detailed reasoning available"}

═══════════════════════════════════════════════════════════════════════

LLM-2 (HISTORICAL PATTERNS):
Conclusion: {llm2_result.get("dominant_historical_motivation", "unknown")} | Confidence: {llm2_result.get("confidence_level", "Low")}
Data Quality: {llm2_result.get("data_quality", "Unknown")}
Reasoning Chain:
{llm2_reasoning if llm2_reasoning.strip() else "  • No detailed reasoning available"}

═══════════════════════════════════════════════════════════════════════

LLM-3 (BEHAVIORAL PATTERNS):
Conclusion: {llm3_result.get("identified_pattern", "unknown")} | Confidence: {llm3_result.get("confidence_level", "Low")}
Pattern Indicators: {', '.join(llm3_result.get("pattern_indicators", []))}
Reasoning Chain:
{llm3_reasoning if llm3_reasoning.strip() else "  • No detailed reasoning available"}

═══════════════════════════════════════════════════════════════════════

FUSION DECISION:
Final Motivation: {fusion_output.get("final_motivation", "unknown")}
Agreement Score: {fusion_output.get("agreement_score", 0)} / 2.85
Final Confidence: {fusion_output.get("final_confidence", "Low")}
Models Aligned: {fusion_output.get("models_agree", False)}
Recommendation: {fusion_output.get("recommendation", "Review required")}

═══════════════════════════════════════════════════════════════════════

NOW GENERATE A COMPREHENSIVE FORENSIC ANALYSIS REPORT THAT:
1. Shows how ALL THREE reasoning chains integrated together
2. Explains where models agreed and why their reasoning aligned
3. For each disagreement, explains the different perspectives
4. Shows the COMPLETE logical flow from evidence → reasoning steps → conclusion
5. Uses specific steps from each model's reasoning chain to build the narrative
6. Explains which reasoning chains were strongest and most influential
7. Provides confident recommendations based on the combined analysis
8. Makes clear how the integrated reasoning led to the final conclusion
"""

        response_text = call_gemini_with_retry(user_prompt, LLM4_SYSTEM_PROMPT, api_key=GEMINI_API_KEY_LLM4)

        return {
            "llm_stage": "LLM-4",
            "status": "success",
            "report": response_text,
            "final_motivation": fusion_output.get("final_motivation"),
            "final_confidence": fusion_output.get("final_confidence"),
            "api_key_used": "LLM4_API_KEY"
        }

    except Exception as e:
        print(f"❌ LLM-4 failed: {e}")
        return {
            "llm_stage": "LLM-4",
            "status": "error",
            "report": f"Failed to generate report: {str(e)[:200]}",
            "final_motivation": fusion_output.get("final_motivation"),
            "final_confidence": fusion_output.get("final_confidence"),
            "api_key_used": "LLM4_API_KEY"
        }

print("✅ Updated LLM agent functions with FULL reasoning chain synthesis for LLM-4")



✅ Updated LLM agent functions with FULL reasoning chain synthesis for LLM-4


In [58]:
def confidence_to_weight(level: str) -> float:
    """Convert confidence level to numerical weight."""
    weights = {"low": 0.4, "medium": 0.65, "high": 0.95}
    return weights.get(str(level).lower(), 0.4)


PATTERN_MOTIVATION_MAP = {
    "domestic_violence_pattern": ("emotional", 0.9),
    "domestic violence pattern": ("emotional", 0.9),

    "public_street_crime_pattern": ("financial", 0.7),
    "public/street crime pattern": ("financial", 0.7),

    "weapon_escalation_pattern": ("financial", 0.8),
    "weapon escalation pattern": ("financial", 0.8),

    "organized_repeat_crime_pattern": ("financial", 0.75),
    "organized/repeat crime pattern": ("financial", 0.75),

    "opportunistic_crime_pattern": ("financial", 0.6),
    "opportunistic crime pattern": ("financial", 0.6),

    "unclear_general_pattern": ("unknown", 0.3),
    "unclear/general pattern": ("unknown", 0.3)
}


def fuse_llm_outputs(llm1: dict, llm2: dict, llm3: dict) -> dict:
    """Fuse outputs from all 3 LLMs with improved semantic accuracy."""

    # -------- Normalization helper (internal only) --------
    def normalize_motivation(m):
        mapping = {
            "financial": "financial",
            "financial gain": "financial",
            "material acquisition": "financial",
            "economic": "financial",

            "power": "power",
            "power and control": "power",
            "dominance": "power",

            "emotional": "emotional",
            "anger": "emotional",
            "revenge": "emotional",

            "unknown": "unknown"
        }
        return mapping.get(str(m).lower(), "unknown")

    # -------- Extract raw values --------
    m1 = normalize_motivation(llm1.get("predicted_motivation", "unknown"))
    m2 = normalize_motivation(llm2.get("dominant_historical_motivation", "unknown"))
    pattern = str(llm3.get("identified_pattern", "unclear/general pattern")).lower()

    c1 = llm1.get("confidence", "Low")
    c2 = llm2.get("confidence_level", "Low")
    c3 = llm3.get("confidence_level", "Low")

    # -------- Weights --------
    w1 = confidence_to_weight(c1)
    w2 = confidence_to_weight(c2)
    w3 = confidence_to_weight(c3)

    # -------- Data quality boost (LLM-2) --------
    data_quality_boost = (
        1.2 if llm2.get("data_quality") == "Exact"
        else 1.1 if llm2.get("data_quality") == "Partial"
        else 1.0
    )
    w2 *= data_quality_boost

    # -------- Pattern-implied motivation --------
    pattern_tuple = PATTERN_MOTIVATION_MAP.get(pattern, ("unknown", 0.3))
    m3, pattern_confidence = pattern_tuple
    m3 = normalize_motivation(m3)
    w3_adjusted = w3 * pattern_confidence

    # -------- Aggregate scores --------
    scores = {}
    contributions = {}

    for motivation, weight, source in [
        (m1, w1, "LLM-1"),
        (m2, w2, "LLM-2"),
        (m3, w3_adjusted, "LLM-3")
    ]:
        if motivation != "unknown" and weight > 0:
            scores[motivation] = scores.get(motivation, 0) + weight
            contributions.setdefault(motivation, []).append(
                (source, round(weight, 3))
            )

    # -------- No usable evidence --------
    if not scores:
        return {
            "final_motivation": "unknown",
            "agreement_score": 0.0,
            "final_confidence": "Low",
            "models_agree": False,
            "conflict_detected": False,
            "supporting_models": {},
            "contribution_breakdown": {},
            "recommendation": "Insufficient evidence"
        }

    # -------- Select winner --------
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    final_motivation, agreement_score = sorted_scores[0]
    second_score = sorted_scores[1][1] if len(sorted_scores) > 1 else 0

    conflict_detected = (agreement_score - second_score) < 0.25

    # -------- Final confidence --------
    if conflict_detected:
        final_confidence = "Low"
    elif agreement_score >= 2.0:
        final_confidence = "High"
    elif agreement_score >= 1.3:
        final_confidence = "Medium"
    else:
        final_confidence = "Low"

    models_agree = len(set([m1, m2, m3]) - {"unknown"}) == 1

    return {
        "final_motivation": final_motivation,
        "agreement_score": round(agreement_score, 3),
        "final_confidence": final_confidence,
        "models_agree": models_agree,
        "conflict_detected": conflict_detected,
        "supporting_models": {
            "LLM-1": {
                "prediction": m1,
                "confidence": c1,
                "contribution": round(w1, 3)
            },
            "LLM-2": {
                "prediction": m2,
                "confidence": c2,
                "contribution": round(w2, 3),
                "data_quality": llm2.get("data_quality")
            },
            "LLM-3": {
                "pattern": pattern,
                "implied_motivation": m3,
                "confidence": c3,
                "contribution": round(w3_adjusted, 3)
            }
        },
        "contribution_breakdown": contributions,
        "recommendation": (
            "High confidence - All models aligned"
            if models_agree and final_confidence == "High"
            else "Review case - Models show disagreement"
            if conflict_detected
            else "Moderate confidence"
        )
    }


print("✅ Improved fusion logic defined (no variable names changed)")


✅ Improved fusion logic defined (no variable names changed)


In [59]:
def print_reasoning_chain_detailed(analysis: dict):
    """Print CHAIN-OF-THOUGHT reasoning from all LLMs with clear formatting."""

    print(f"\n{'='*80}")
    print("🧠 CHAIN-OF-THOUGHT REASONING (DETAILED)")
    print(f"{'='*80}\n")

    # LLM-1 Reasoning Chain
    print(f"{'─'*80}")
    print("🔍 LLM-1: MOTIVATION ANALYSIS - CHAIN-OF-THOUGHT")
    print(f"{'─'*80}")
    llm1 = analysis['stage_1_motivation_analysis']
    reasoning_chain = llm1.get('reasoning_chain', [])

    if reasoning_chain and len(reasoning_chain) > 0:
        print(f"\n{len(reasoning_chain)} Reasoning Steps:\n")
        for i, step in enumerate(reasoning_chain, 1):
            print(f"  Step {i}: {step}\n")
    else:
        print("  ⚠️  No detailed reasoning chain returned\n")

    print(f"  📊 FINAL PREDICTION: {llm1['predicted_motivation'].upper()}")
    print(f"  🎯 Confidence: {llm1['confidence']}")
    print(f"  💭 Summary: {llm1['reasoning'][:150]}...\n")

    # LLM-2 Reasoning Chain
    print(f"{'─'*80}")
    print("📊 LLM-2: HISTORICAL ANALYSIS - CHAIN-OF-THOUGHT")
    print(f"{'─'*80}")
    llm2 = analysis['stage_2_historical_analysis']
    reasoning_chain = llm2.get('reasoning_chain', [])

    if reasoning_chain and len(reasoning_chain) > 0:
        print(f"\n{len(reasoning_chain)} Reasoning Steps:\n")
        for i, step in enumerate(reasoning_chain, 1):
            print(f"  Step {i}: {step}\n")
    else:
        print("  ⚠️  No detailed reasoning chain returned\n")

    print(f"  📊 FINAL PREDICTION: {llm2['dominant_historical_motivation'].upper()}")
    print(f"  🎯 Confidence: {llm2['confidence_level']}")
    print(f"  📈 Data Quality: {llm2['data_quality']}\n")

    # LLM-3 Reasoning Chain
    print(f"{'─'*80}")
    print("🎭 LLM-3: PATTERN ANALYSIS - CHAIN-OF-THOUGHT")
    print(f"{'─'*80}")
    llm3 = analysis['stage_3_pattern_analysis']
    reasoning_chain = llm3.get('reasoning_chain', [])

    if reasoning_chain and len(reasoning_chain) > 0:
        print(f"\n{len(reasoning_chain)} Reasoning Steps:\n")
        for i, step in enumerate(reasoning_chain, 1):
            print(f"  Step {i}: {step}\n")
    else:
        print("  ⚠️  No detailed reasoning chain returned\n")

    print(f"  🎯 IDENTIFIED PATTERN: {llm3['identified_pattern'].upper()}")
    print(f"  🎯 Confidence: {llm3['confidence_level']}")
    print(f"  📍 Pattern Indicators: {llm3['pattern_indicators']}\n")

    # Fusion Result
    print(f"{'─'*80}")
    print("⚡ FUSION LAYER: COMBINED DECISION")
    print(f"{'─'*80}")
    fusion = analysis['fusion_layer']
    print(f"\n  Final Motivation: {fusion['final_motivation'].upper()}")
    print(f"  Agreement Score: {fusion['agreement_score']}/2.85")
    print(f"  Final Confidence: {fusion['final_confidence']}")
    print(f"  Models Aligned: {fusion['models_agree']}")
    print(f"  Recommendation: {fusion['recommendation']}\n")
    print(f"{'='*80}\n")


print("✅ Chain-of-Thought display function defined")


✅ Chain-of-Thought display function defined


In [60]:
def analyze_crime_case(crime_record: dict, verbose: bool = True, show_cot: bool = True) -> dict:
    """Complete end-to-end analysis of a crime case with INTEGRATED reasoning synthesis."""

    # Safety check: Make sure MODELS is loaded
    if 'MODELS' not in globals() or MODELS is None:
        print("❌ ERROR: Models not loaded!")
        print("\n📌 FIX: You must run the '📦 LOAD PRE-TRAINED MODELS' cell first")
        print("\nCell order:")
        print("  1. Install dependencies")
        print("  2. Upload model files")
        print("  3. Configure API keys")
        print("  4. Import libraries")
        print("  5. Configure Gemini API")
        print("  6. System prompts")
        print("  7. ⭐ LOAD PRE-TRAINED MODELS ← RUN THIS CELL")
        print("  8. LLM agent functions")
        print("  9. Fusion logic")
        print("  10. Main pipeline functions")
        print("  11. Quick analysis")
        raise RuntimeError("MODELS not initialized. Run the model loading cell first.")

    if verbose:
        print(f"\n{'='*70}")
        print("🔬 ANALYZING CRIME CASE")
        print(f"{'='*70}")

    # LLM-1
    if verbose:
        print("\n[LLM-1] Analyzing crime motivation...")
    llm1_result = llm1_agent_from_preprocessed(crime_record)
    if verbose:
        print(f"  → {llm1_result['predicted_motivation']} ({llm1_result['confidence']})")

    # LLM-2
    if verbose:
        print("\n[LLM-2] Analyzing historical patterns...")
    llm2_context = {
        "state": crime_record.get("area_name", "Unknown"),
        "year": 2020,
        "crime_type": crime_record.get("crm_cd_desc", "Unknown")
    }
    llm2_result = llm2_agent_from_preprocessed(llm2_context, MODELS["llm2_dataset"])
    if verbose:
        print(f"  → {llm2_result['dominant_historical_motivation']} ({llm2_result['confidence_level']})")

    # LLM-3
    if verbose:
        print("\n[LLM-3] Identifying behavioral patterns...")
    llm3_record = {
        "context_text": crime_record.get("crime_text", ""),
        "crime_type": crime_record.get("crm_cd_desc", "Unknown"),
        "location_type": crime_record.get("premis_desc", "Unknown"),
        "domestic": crime_record.get("domestic", "Unknown"),
        "arrest": crime_record.get("status_desc", "Unknown"),
        "district": "Unknown",
        "year": 2020
    }
    llm3_result = llm3_agent_from_preprocessed(llm3_record)
    if verbose:
        print(f"  → {llm3_result['identified_pattern']}")

    # Fusion
    if verbose:
        print("\n[FUSION] Combining all analyses...")
    fusion_result = fuse_llm_outputs(llm1_result, llm2_result, llm3_result)
    if verbose:
        print(f"  → Final: {fusion_result['final_motivation']} ({fusion_result['final_confidence']})")
        print(f"  → Score: {fusion_result['agreement_score']}")

    # LLM-4 - NOW RECEIVES ALL REASONING CHAINS
    if verbose:
        print("\n[LLM-4] Synthesizing complete integrated analysis...")
    llm4_result = llm4_generate_report(fusion_result, llm1_result, llm2_result, llm3_result, crime_record)

    analysis = {
        "case_id": f"CASE-{hash(str(crime_record)) % 10000:05d}",
        "timestamp": pd.Timestamp.now().isoformat(),
        "crime_details": crime_record,
        "stage_1_motivation_analysis": llm1_result,
        "stage_2_historical_analysis": llm2_result,
        "stage_3_pattern_analysis": llm3_result,
        "fusion_layer": fusion_result,
        "stage_4_report": llm4_result,
        "analysis_summary": {
            "final_motivation": fusion_result['final_motivation'],
            "final_confidence": fusion_result['final_confidence'],
            "recommendation": fusion_result['recommendation'],
            "models_agreed": fusion_result['models_agree'],
            "conflict_detected": fusion_result['conflict_detected']
        }
    }

    # AUTOMATICALLY DISPLAY COMPLETE ANALYSIS
    if show_cot:
        print_reasoning_chain_detailed(analysis)
        print_llm4_integrated_report(analysis)
        print_case_summary(analysis)

    return analysis


def print_case_summary(analysis: dict):
    """Print formatted case summary."""
    summary = analysis['analysis_summary']
    fusion = analysis['fusion_layer']

    print(f"\n{'='*70}")
    print("📋 ANALYSIS SUMMARY")
    print(f"{'='*70}")
    print(f"Case: {analysis['case_id']}")
    print(f"\n🎯 FINAL DETERMINATION:")
    print(f"   Motivation: {summary['final_motivation'].upper()}")
    print(f"   Confidence: {summary['final_confidence']}")
    print(f"   Agreement Score: {fusion['agreement_score']}/2.85")
    print(f"\n📊 MODEL CONSENSUS:")
    print(f"   All aligned: {summary['models_agreed']}")
    print(f"   Recommendation: {summary['recommendation']}")
    print(f"{'='*70}\n")


def print_llm4_integrated_report(analysis: dict):
    """Print LLM-4 comprehensive integrated analysis report."""
    llm4 = analysis['stage_4_report']

    print(f"\n{'='*80}")
    print("📊 LLM-4: INTEGRATED FORENSIC ANALYSIS REPORT")
    print(f"{'='*80}\n")

    if llm4.get("status") == "success":
        print(llm4.get("report", "No report generated"))
    else:
        print(f"⚠️  {llm4.get('report', 'Report generation failed')}")

    print(f"\n{'='*80}\n")


print("✅ Pipeline functions updated with INTEGRATED reasoning synthesis + safety checks")



✅ Pipeline functions updated with INTEGRATED reasoning synthesis + safety checks


In [61]:
def load_models():
    """Load all pre-trained models and datasets."""
    # ... function code ...

# Load models
MODELS = load_models()
print("\n✅ All models loaded successfully!")


✅ All models loaded successfully!


In [62]:
# ── Safety check: auto-load models if not in memory ──────────────────
import joblib, json, torch
from sentence_transformers import SentenceTransformer

if 'MODELS' not in globals() or MODELS is None:
    print("⚠️  MODELS not loaded — loading now...\n")

    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    device_name = "🔴 GPU (CUDA)" if torch.cuda.is_available() else "🟢 CPU"

    llm1_embedder   = SentenceTransformer("/content/llm1_model/embedder")
    llm1_classifier = joblib.load("/content/llm1_model/classifier.joblib")
    with open("/content/llm1_model/metadata.json") as f:
        llm1_metadata = json.load(f)

    llm2_model         = joblib.load("/content/llm2_model/model.joblib")
    llm2_state_encoder = joblib.load("/content/llm2_model/state_encoder.joblib")
    with open("/content/llm2_model/metadata.json") as f:
        llm2_metadata = json.load(f)

    llm3_embedder = SentenceTransformer("/content/llm3_model/embedder")
    llm3_kmeans   = joblib.load("/content/llm3_model/kmeans.joblib")
    with open("/content/llm3_model/metadata.json") as f:
        llm3_metadata = json.load(f)

    with open("/content/llm2_motivation_dataset.json", "r", encoding="utf-8") as f:
        llm2_dataset = json.load(f)

    with open("/content/llm3_clustered_data.json", "r", encoding="utf-8") as f:
        llm3_dataset = json.load(f)

    MODELS = {
        "llm1_embedder":      llm1_embedder,
        "llm1_classifier":    llm1_classifier,
        "llm1_metadata":      llm1_metadata,
        "llm2_model":         llm2_model,
        "llm2_state_encoder": llm2_state_encoder,
        "llm2_metadata":      llm2_metadata,
        "llm2_dataset":       llm2_dataset,
        "llm3_embedder":      llm3_embedder,
        "llm3_kmeans":        llm3_kmeans,
        "llm3_metadata":      llm3_metadata,
        "llm3_dataset":       llm3_dataset,
        "device":             device,
        "device_name":        device_name
    }
    print("✅ MODELS loaded successfully!\n")

# ── Demonstration ─────────────────────────────────────────────────────
print("\n" + "="*70)
print("🔍 CRIME MOTIVATION ANALYSIS PIPELINE - DEMONSTRATION")
print("="*70)

sample_crimes = [
    {
        "crime_text": "On 2020-05-10 at 22 hours, in central area, a 21-year-old M was involved in robbery at street. Weapon used: handgun. Case status: investigation continuing.",
        "crm_cd_desc": "robbery",
        "area_name": "central",
        "premis_desc": "street",
        "vict_age": "21",
        "vict_sex": "M",
        "weapon_desc": "handgun",
        "status_desc": "invest cont",
        "domestic": "false"
    },
    {
        "crime_text": "On 2020-08-15 at 19 hours, in residential area, a 35-year-old F reported domestic violence by intimate partner. Weapon used: hands/fists. Case status: reported.",
        "crm_cd_desc": "domestic violence",
        "area_name": "residential",
        "premis_desc": "home",
        "vict_age": "35",
        "vict_sex": "F",
        "weapon_desc": "hands/fists",
        "status_desc": "reported",
        "domestic": "true"
    }
]

print(f"\n🧪 DEMONSTRATION\n")
print(f"Analyzing {len(sample_crimes)} sample crimes...\n")

print("\n📍 CASE 1: ROBBERY\n")
result1 = analyze_crime_case(sample_crimes[0], verbose=True)
print_reasoning_chain_detailed(result1)
print_case_summary(result1)

print("\n📍 CASE 2: DOMESTIC VIOLENCE\n")
result2 = analyze_crime_case(sample_crimes[1], verbose=True)
print_reasoning_chain_detailed(result2)
print_case_summary(result2)

print("\n✅ PIPELINE DEMONSTRATION COMPLETE!")

⚠️  MODELS not loaded — loading now...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ MODELS loaded successfully!


🔍 CRIME MOTIVATION ANALYSIS PIPELINE - DEMONSTRATION

🧪 DEMONSTRATION

Analyzing 2 sample crimes...


📍 CASE 1: ROBBERY


🔬 ANALYZING CRIME CASE

[LLM-1] Analyzing crime motivation...
  → financial (High)

[LLM-2] Analyzing historical patterns...
  → Financial Gain (economic necessity or greed) (Medium)

[LLM-3] Identifying behavioral patterns...
  → public_street_crime_pattern

[FUSION] Combining all analyses...
  → Final: financial (Medium)
  → Score: 1.615

[LLM-4] Synthesizing complete integrated analysis...

🧠 CHAIN-OF-THOUGHT REASONING (DETAILED)

────────────────────────────────────────────────────────────────────────────────
🔍 LLM-1: MOTIVATION ANALYSIS - CHAIN-OF-THOUGHT
────────────────────────────────────────────────────────────────────────────────

6 Reasoning Steps:

  Step 1: EXTRACT KEY FACTS: The core fact is the 'Crime Type: robbery' and the 'Weapon: handgun'.

  Step 2: IDENTIFY INDICATORS: The term 'robbery' directly indicates an act a

In [63]:
import joblib, json, torch
from sentence_transformers import SentenceTransformer

if 'MODELS' not in globals() or MODELS is None:
    print("⚠️  MODELS not loaded — loading now...\n")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    llm1_embedder   = SentenceTransformer("/content/llm1_model/embedder")
    llm1_classifier = joblib.load("/content/llm1_model/classifier.joblib")
    with open("/content/llm1_model/metadata.json") as f: llm1_metadata = json.load(f)
    llm2_model         = joblib.load("/content/llm2_model/model.joblib")
    llm2_state_encoder = joblib.load("/content/llm2_model/state_encoder.joblib")
    with open("/content/llm2_model/metadata.json") as f: llm2_metadata = json.load(f)
    llm3_embedder = SentenceTransformer("/content/llm3_model/embedder")
    llm3_kmeans   = joblib.load("/content/llm3_model/kmeans.joblib")
    with open("/content/llm3_model/metadata.json") as f: llm3_metadata = json.load(f)
    with open("/content/llm2_motivation_dataset.json", "r", encoding="utf-8") as f: llm2_dataset = json.load(f)
    with open("/content/llm3_clustered_data.json", "r", encoding="utf-8") as f: llm3_dataset = json.load(f)
    MODELS = {
        "llm1_embedder": llm1_embedder, "llm1_classifier": llm1_classifier, "llm1_metadata": llm1_metadata,
        "llm2_model": llm2_model, "llm2_state_encoder": llm2_state_encoder, "llm2_metadata": llm2_metadata,
        "llm2_dataset": llm2_dataset, "llm3_embedder": llm3_embedder, "llm3_kmeans": llm3_kmeans,
        "llm3_metadata": llm3_metadata, "llm3_dataset": llm3_dataset,
        "device": device, "device_name": str(device)
    }
    print("✅ MODELS loaded!\n")

# ── Define all crimes ──────────────────────────────────────────────────
crimes = {
    "ROBBERY": {
        "crime_text": "On 2020-05-10 at 22 hours, robbed a convenience store with handgun. Demanded money from cashier.",
        "crm_cd_desc": "robbery", "area_name": "downtown", "premis_desc": "store",
        "vict_age": "28", "vict_sex": "M", "weapon_desc": "handgun",
        "status_desc": "arrested", "domestic": "false"
    },
    "DOMESTIC VIOLENCE": {
        "crime_text": "Intimate partner assault. Subject attacked with fists during argument about finances. Victim had visible injuries.",
        "crm_cd_desc": "domestic violence", "area_name": "residential", "premis_desc": "home",
        "vict_age": "35", "vict_sex": "F", "weapon_desc": "hands/fists",
        "status_desc": "reported", "domestic": "true"
    },
    "ASSAULT": {
        "crime_text": "Street fight between two groups. Subject struck victim with metal pipe after heated argument.",
        "crm_cd_desc": "assault", "area_name": "central", "premis_desc": "street",
        "vict_age": "32", "vict_sex": "M", "weapon_desc": "pipe",
        "status_desc": "charged", "domestic": "false"
    },
    "MURDER": {
        "crime_text": "Victim found deceased with multiple stab wounds. Crime scene suggests premeditation. Suspect arrested at scene.",
        "crm_cd_desc": "murder", "area_name": "downtown", "premis_desc": "alley",
        "vict_age": "45", "vict_sex": "M", "weapon_desc": "knife",
        "status_desc": "arrested", "domestic": "false"
    },
    "SEXUAL ASSAULT": {
        "crime_text": "Victim reported unwanted sexual contact by acquaintance at party. Victim did not consent. Medical evidence collected.",
        "crm_cd_desc": "sexual assault", "area_name": "uptown", "premis_desc": "residence",
        "vict_age": "22", "vict_sex": "F", "weapon_desc": "none",
        "status_desc": "investigation", "domestic": "false"
    }
}

# ── Analyze all crimes ─────────────────────────────────────────────────
for crime_type, crime_record in crimes.items():
    print(f"\n{'='*70}")
    print(f"📍 CASE: {crime_type}")
    print(f"{'='*70}")
    result = analyze_crime_case(crime_record, verbose=True)
    print_case_summary(result)

print("\n✅ ALL CASES ANALYZED!")




📍 CASE: ROBBERY

🔬 ANALYZING CRIME CASE

[LLM-1] Analyzing crime motivation...
  → financial (High)

[LLM-2] Analyzing historical patterns...
  → Financial Gain (Medium)

[LLM-3] Identifying behavioral patterns...
  → weapon_escalation_pattern

[FUSION] Combining all analyses...
  → Final: financial (High)
  → Score: 2.36

[LLM-4] Synthesizing complete integrated analysis...

🧠 CHAIN-OF-THOUGHT REASONING (DETAILED)

────────────────────────────────────────────────────────────────────────────────
🔍 LLM-1: MOTIVATION ANALYSIS - CHAIN-OF-THOUGHT
────────────────────────────────────────────────────────────────────────────────

6 Reasoning Steps:

  Step 1: EXTRACT KEY FACTS: The core actions are 'robbed a convenience store' and 'demanded money from cashier', with a 'handgun' used.

  Step 2: IDENTIFY INDICATORS: 'Robbed a convenience store' indicates an act of taking property, typically money or goods. 'Demanded money from cashier' explicitly states the objective was monetary acquisition.

⚠️ Gemini API error (attempt 1/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2
  → weapon_escalation_pattern

[FUSION] Combining all analyses...
  → Final: emotional (Low)
  → Score: 0.95

[LLM-4] Synthesizing complete integrated analysis...

🧠 CHAIN-OF-THOUGHT REASONING (DETAILED)

────────────────────────────────────────────────────────────────────────────────
🔍 LLM-1: MOTIVATION ANALYSIS - CHAIN-OF-THOUGHT
────────────────────────────────────────────────────────────────────────────────

6 Reasoning Steps:

  Step 1: EXTRACT KEY FACTS: Identified key details including 'multiple stab wounds,' 'premeditation,' 'knife' as weapon, and 'suspect arrested at scene.'

  Step 2: IDENTIFY INDICATORS: 'Multiple stab wounds' indicates potential overkill, intense rage, or a personal vendetta. 'Premeditation' suggests a planned act, not spontaneous. 'Knife' is a close-contact, personal weapon.

  Step 3: MAP TO MOTIVATIONS: 'Multiple stab wo

In [48]:
#STEP 2 — LOAD LLM-2 MOTIVATION JSON
with open("/content/llm2_motivation_dataset.json", "r") as f:
    llm2_data = json.load(f)

# Convert motivation JSON to fast lookup
motivation_lookup = {
    (item["state"], int(item["year"])): sum(item["motivation_distribution"].values())
    for item in llm2_data
}

print("✅ Motivation lookup ready")


✅ Motivation lookup ready


In [40]:
# =========================
# LLM-4 METRICS
# =========================

def run_llm4_with_metrics(user_input):
    print("\n==============================")
    print("🚀 RUNNING LLM-4 WITH METRICS")
    print("==============================\n")

    response, latency = _measure_latency(llm4, user_input)

    metrics = {
        "latency_ms": latency,
        "response_length": len(response),
        "confidence_score": _confidence_score(llm4, response),
        "quality_score": _quality_score(llm4, response, user_input),
    }

    response_2 = llm4(user_input)
    metrics["consistency_score"] = _consistency_score(response, response_2)

    metrics["final_score"] = _final_score(metrics)

    print("📤 LLM-4 OUTPUT:\n")
    print(response)

    print("\n📊 LLM-4 METRICS:\n")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    print(f"\n🏁 LLM-4 FINAL SCORE: {metrics['final_score']}/100")
    print("\n==============================\n")

    return response, metrics


# GUI


In [64]:
from IPython.display import display, HTML

def show_report(result, llm1_metadata, llm2_metadata, llm3_metadata):
    """Display full forensic analysis report as GUI in Colab."""

    # ── Pull metadata ──────────────────────────────────────────────────
    acc      = llm1_metadata.get("accuracy", 0)
    f1       = llm1_metadata.get("cv_mean_f1", 0)
    labels   = llm1_metadata.get("labels", [])
    mae      = llm2_metadata.get("mae", 0)
    mot_cols = llm2_metadata.get("motivation_columns", [])
    clusters = llm3_metadata.get("num_clusters", 0)
    records  = llm3_metadata.get("total_records", 0)

    # ── Pull results ───────────────────────────────────────────────────
    llm1   = result["stage_1_motivation_analysis"]
    llm2   = result["stage_2_historical_analysis"]
    llm3   = result["stage_3_pattern_analysis"]
    fusion = result["fusion_layer"]
    llm4   = result["stage_4_report"]
    summary = result["analysis_summary"]
    case_id = result["case_id"]

    # ── Helpers ────────────────────────────────────────────────────────
    def conf_color(c):
        c = str(c).lower()
        if c == 'high':   return '#39ff14'
        if c == 'medium': return '#ffb800'
        return '#ff4b6e'

    def chain_html(steps, color):
        if not steps:
            return '<div style="color:#4a6070;font-size:0.7rem;">No reasoning chain available</div>'
        rows = ""
        for i, step in enumerate(steps, 1):
            rows += f'<div class="step"><span class="snum" style="color:{color};">0{i}</span><span class="stext">{step}</span></div>'
        return rows

    tags_llm1 = ''.join(f'<span class="tag">{l}</span>' for l in labels)
    tags_llm2 = ''.join(f'<span class="tag">{c}</span>' for c in mot_cols[:4])
    tags_llm2 += '<span class="tag">+more</span>' if len(mot_cols) > 4 else ''
    mae_bar   = max(0, 100 - float(mae) * 10)
    cols_bar  = min(100, len(mot_cols) * 10)
    clust_bar = min(100, clusters * 20)
    rec_bar   = min(100, records / 100)
    gauge_pct = (fusion['agreement_score'] / 2.85) * 100

    html = f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Orbitron:wght@700;900&family=Rajdhani:wght@400;600&display=swap');
  .cg * {{ box-sizing:border-box; margin:0; padding:0; }}
  .cg {{
    background:#090b0f; color:#c8d8e8;
    font-family:'Share Tech Mono',monospace;
    padding:24px; border:1px solid #1e2d3d;
    position:relative;
  }}

  /* HEADER */
  .cg-header {{
    display:flex; justify-content:space-between; align-items:center;
    border-bottom:1px solid #1e2d3d; padding-bottom:14px; margin-bottom:20px;
  }}
  .cg-logo {{ font-family:'Orbitron',monospace; font-size:1.1rem; font-weight:900; color:#00d4ff; letter-spacing:0.15em; text-shadow:0 0 20px rgba(0,212,255,0.5); }}
  .cg-sub {{ font-size:0.55rem; color:#4a6070; letter-spacing:0.2em; margin-top:2px; }}
  .cg-caseid {{ font-size:0.6rem; color:#4a6070; letter-spacing:0.1em; text-align:right; }}
  .cg-badge {{ font-size:0.58rem; padding:3px 8px; border:1px solid; display:inline-flex; align-items:center; gap:5px; margin-left:8px; }}
  .cg-badge.on {{ border-color:#39ff14; color:#39ff14; }}
  .cg-dot {{ width:5px; height:5px; border-radius:50%; background:currentColor; animation:cgblink 1.5s infinite; }}
  @keyframes cgblink {{ 0%,100%{{opacity:1}} 50%{{opacity:0.2}} }}

  /* SECTION TITLE */
  .cg-stitle {{ font-family:'Orbitron',monospace; font-size:0.65rem; letter-spacing:0.2em; color:#00d4ff; margin-bottom:12px; display:flex; justify-content:space-between; }}
  .cg-stitle span {{ font-size:0.55rem; color:#4a6070; }}

  /* METRICS */
  .cg-metrics {{ display:grid; grid-template-columns:repeat(3,1fr); gap:12px; margin-bottom:22px; }}
  .cg-mcard {{ background:#0d1117; border:1px solid #1e2d3d; padding:14px; position:relative; overflow:hidden; }}
  .cg-mcard::after {{ content:''; position:absolute; bottom:0; left:0; right:0; height:2px; }}
  .cg-mcard.m1::after {{ background:#00d4ff; }}
  .cg-mcard.m2::after {{ background:#ffb800; }}
  .cg-mcard.m3::after {{ background:#39ff14; }}
  .cg-mname {{ font-family:'Orbitron',monospace; font-size:0.9rem; font-weight:700; margin-bottom:2px; }}
  .cg-mcard.m1 .cg-mname {{ color:#00d4ff; }}
  .cg-mcard.m2 .cg-mname {{ color:#ffb800; }}
  .cg-mcard.m3 .cg-mname {{ color:#39ff14; }}
  .cg-mrole {{ font-size:0.55rem; color:#4a6070; letter-spacing:0.12em; text-transform:uppercase; margin-bottom:12px; }}
  .cg-mrow {{ display:flex; justify-content:space-between; margin-bottom:7px; align-items:center; }}
  .cg-mk {{ font-size:0.57rem; color:#4a6070; text-transform:uppercase; }}
  .cg-mv {{ font-size:0.8rem; font-weight:700; }}
  .cg-mcard.m1 .cg-mv {{ color:#00d4ff; }}
  .cg-mcard.m2 .cg-mv {{ color:#ffb800; }}
  .cg-mcard.m3 .cg-mv {{ color:#39ff14; }}
  .cg-bwrap {{ height:3px; background:#1e2d3d; margin-bottom:10px; overflow:hidden; }}
  .cg-bar {{ height:100%; transition:width 1.5s cubic-bezier(0.4,0,0.2,1); width:0; }}
  .cg-mcard.m1 .cg-bar {{ background:#00d4ff; }}
  .cg-mcard.m2 .cg-bar {{ background:#ffb800; }}
  .cg-mcard.m3 .cg-bar {{ background:#39ff14; }}
  .cg-mnote {{ font-size:0.55rem; color:#4a6070; line-height:1.6; border-top:1px solid #1e2d3d; padding-top:8px; margin-top:6px; }}
  .tags {{ display:flex; flex-wrap:wrap; gap:4px; margin-top:8px; }}
  .tag {{ font-size:0.52rem; padding:2px 6px; border:1px solid #1e2d3d; color:#4a6070; }}

  /* VERDICT */
  .cg-verdict {{ background:#0d1117; border:1px solid rgba(255,75,110,0.3); padding:20px; text-align:center; margin-bottom:20px; position:relative; }}
  .cg-vmot {{ font-family:'Orbitron',monospace; font-size:2rem; font-weight:900; color:#ff4b6e; letter-spacing:0.15em; text-shadow:0 0 30px rgba(255,75,110,0.4); }}
  .cg-vsub {{ font-size:0.6rem; color:#4a6070; margin-top:10px; letter-spacing:0.1em; }}
  .cg-gauge {{ display:flex; align-items:center; gap:12px; margin-top:14px; }}
  .cg-gtrack {{ flex:1; height:6px; background:#1e2d3d; overflow:hidden; }}
  .cg-gfill {{ height:100%; background:linear-gradient(90deg,#ff4b6e,#00d4ff); transition:width 1.5s cubic-bezier(0.4,0,0.2,1); width:0; }}
  .cg-glabel {{ font-size:0.58rem; color:#4a6070; min-width:40px; }}

  /* STAGES */
  .cg-stage {{ border:1px solid #1e2d3d; margin-bottom:10px; }}
  .cg-shead {{ padding:10px 14px; display:flex; justify-content:space-between; align-items:center; cursor:pointer; user-select:none; }}
  .cg-shead:hover {{ filter:brightness(1.3); }}
  .cg-stage.s1 .cg-shead {{ background:rgba(0,212,255,0.05); border-bottom:1px solid rgba(0,212,255,0.12); }}
  .cg-stage.s2 .cg-shead {{ background:rgba(255,184,0,0.05); border-bottom:1px solid rgba(255,184,0,0.12); }}
  .cg-stage.s3 .cg-shead {{ background:rgba(57,255,20,0.05); border-bottom:1px solid rgba(57,255,20,0.12); }}
  .cg-stage.sf .cg-shead {{ background:rgba(255,75,110,0.05); border-bottom:1px solid rgba(255,75,110,0.12); }}
  .cg-stage.s4 .cg-shead {{ background:rgba(0,212,255,0.03); border-bottom:1px solid rgba(0,212,255,0.08); }}
  .cg-stag {{ font-family:'Orbitron',monospace; font-size:0.68rem; font-weight:700; letter-spacing:0.1em; }}
  .cg-stage.s1 .cg-stag {{ color:#00d4ff; }}
  .cg-stage.s2 .cg-stag {{ color:#ffb800; }}
  .cg-stage.s3 .cg-stag {{ color:#39ff14; }}
  .cg-stage.sf .cg-stag {{ color:#ff4b6e; }}
  .cg-stage.s4 .cg-stag {{ color:#00d4ff; }}
  .cg-sbadge {{ font-size:0.6rem; padding:2px 8px; border:1px solid; }}
  .cg-sbody {{ padding:14px; display:none; }}
  .cg-sbody.open {{ display:block; }}
  .kv {{ display:flex; gap:10px; margin-bottom:7px; }}
  .kk {{ color:#4a6070; min-width:140px; flex-shrink:0; font-size:0.6rem; text-transform:uppercase; }}
  .vv {{ color:#c8d8e8; font-size:0.7rem; line-height:1.6; }}
  .cg-div {{ height:1px; background:#1e2d3d; margin:10px 0; }}
  .chain-label {{ font-size:0.57rem; color:#4a6070; letter-spacing:0.12em; margin-bottom:8px; }}
  .step {{ display:flex; gap:10px; margin-bottom:8px; }}
  .snum {{ font-family:'Orbitron',monospace; font-size:0.58rem; min-width:22px; margin-top:2px; }}
  .stext {{ font-size:0.68rem; line-height:1.6; color:#c8d8e8; }}
  .cg-report {{ white-space:pre-wrap; font-family:'Rajdhani',sans-serif; font-size:0.88rem; line-height:1.9; color:#c8d8e8; background:#0a0e14; padding:16px; border-left:3px solid #00d4ff; }}
</style>

<div class="cg">

  <!-- HEADER -->
  <div class="cg-header">
    <div>
      <div class="cg-logo">CRIMEMIND</div>
      <div class="cg-sub">CRIME MOTIVATION ANALYSIS SYSTEM v2.0</div>
    </div>
    <div style="text-align:right;">
      <div class="cg-caseid">{case_id}</div>
      <div class="cg-caseid" style="margin-top:4px;">{result['timestamp'][:19]}</div>
      <div style="margin-top:6px;">
        <span class="cg-badge on"><span class="cg-dot"></span>SYSTEM ONLINE</span>
      </div>
    </div>
  </div>

  <!-- METRICS -->
  <div class="cg-stitle">⬡ MODEL PERFORMANCE METRICS</div>
  <div class="cg-metrics">
    <div class="cg-mcard m1">
      <div class="cg-mname">LLM-1</div>
      <div class="cg-mrole">Motivation Classifier</div>
      <div class="cg-mrow"><span class="cg-mk">Accuracy</span><span class="cg-mv">{acc*100:.1f}%</span></div>
      <div class="cg-bwrap"><div class="cg-bar" id="cgb1"></div></div>
      <div class="cg-mrow"><span class="cg-mk">CV F1 Score</span><span class="cg-mv">{f1*100:.1f}%</span></div>
      <div class="cg-bwrap"><div class="cg-bar" id="cgb2"></div></div>
      <div class="cg-mnote">LogisticRegression · all-MiniLM-L6-v2 · 5-fold CV</div>
      <div class="tags">{tags_llm1}</div>
    </div>
    <div class="cg-mcard m2">
      <div class="cg-mname">LLM-2</div>
      <div class="cg-mrole">Historical Analyzer</div>
      <div class="cg-mrow"><span class="cg-mk">Mean Abs Error</span><span class="cg-mv">{mae}</span></div>
      <div class="cg-bwrap"><div class="cg-bar" id="cgb3"></div></div>
      <div class="cg-mrow"><span class="cg-mk">Motivation Cols</span><span class="cg-mv">{len(mot_cols)}</span></div>
      <div class="cg-bwrap"><div class="cg-bar" id="cgb4"></div></div>
      <div class="cg-mnote">MultiOutputRegressor · RandomForest(200)</div>
      <div class="tags">{tags_llm2}</div>
    </div>
    <div class="cg-mcard m3">
      <div class="cg-mname">LLM-3</div>
      <div class="cg-mrole">Pattern Clusterer</div>
      <div class="cg-mrow"><span class="cg-mk">Clusters</span><span class="cg-mv">{clusters}</span></div>
      <div class="cg-bwrap"><div class="cg-bar" id="cgb5"></div></div>
      <div class="cg-mrow"><span class="cg-mk">Total Records</span><span class="cg-mv">{records:,}</span></div>
      <div class="cg-bwrap"><div class="cg-bar" id="cgb6"></div></div>
      <div class="cg-mnote">MiniBatchKMeans(k={clusters}) · all-MiniLM-L6-v2</div>
    </div>
  </div>

  <!-- VERDICT -->
  <div class="cg-stitle">⬡ CASE ANALYSIS <span>{case_id} · {result['timestamp'][:19]}</span></div>
  <div class="cg-verdict">
    <div style="font-size:0.55rem;color:#4a6070;letter-spacing:0.2em;margin-bottom:8px;">FINAL DETERMINATION</div>
    <div class="cg-vmot">{summary['final_motivation'].upper()}</div>
    <div class="cg-vsub">
      CONFIDENCE: {summary['final_confidence'].upper()} &nbsp;|&nbsp;
      MODELS ALIGNED: {summary['models_agreed']} &nbsp;|&nbsp;
      SCORE: {fusion['agreement_score']}/2.85 &nbsp;|&nbsp;
      {summary['recommendation']}
    </div>
    <div class="cg-gauge">
      <div class="cg-glabel">SCORE</div>
      <div class="cg-gtrack"><div class="cg-gfill" id="cggauge"></div></div>
      <div class="cg-glabel" style="text-align:right">{fusion['agreement_score']}/2.85</div>
    </div>
  </div>

  <!-- LLM-1 -->
  <div class="cg-stage s1">
    <div class="cg-shead" onclick="this.nextElementSibling.classList.toggle('open')">
      <div><span class="cg-stag">LLM-1</span><span style="font-size:0.57rem;color:#4a6070;margin-left:8px;">MOTIVATION CLASSIFIER</span></div>
      <span class="cg-sbadge" style="border-color:{conf_color(llm1['confidence'])};color:{conf_color(llm1['confidence'])}">{llm1['predicted_motivation'].upper()} · {llm1['confidence']}</span>
    </div>
    <div class="cg-sbody open">
      <div class="kv"><span class="kk">Prediction</span><span class="vv" style="color:#00d4ff;text-transform:uppercase;">{llm1['predicted_motivation']}</span></div>
      <div class="kv"><span class="kk">Confidence</span><span class="vv" style="color:{conf_color(llm1['confidence'])}">{llm1['confidence']}</span></div>
      <div class="kv"><span class="kk">Reasoning</span><span class="vv">{str(llm1.get('reasoning','—'))[:250]}...</span></div>
      <div class="cg-div"></div>
      <div class="chain-label">CHAIN-OF-THOUGHT STEPS</div>
      {chain_html(llm1.get('reasoning_chain', []), '#00d4ff')}
    </div>
  </div>

  <!-- LLM-2 -->
  <div class="cg-stage s2">
    <div class="cg-shead" onclick="this.nextElementSibling.classList.toggle('open')">
      <div><span class="cg-stag">LLM-2</span><span style="font-size:0.57rem;color:#4a6070;margin-left:8px;">HISTORICAL ANALYZER</span></div>
      <span class="cg-sbadge" style="border-color:{conf_color(llm2['confidence_level'])};color:{conf_color(llm2['confidence_level'])}">{llm2['dominant_historical_motivation'].upper()} · {llm2['confidence_level']}</span>
    </div>
    <div class="cg-sbody">
      <div class="kv"><span class="kk">Historical Pattern</span><span class="vv" style="color:#ffb800;text-transform:uppercase;">{llm2['dominant_historical_motivation']}</span></div>
      <div class="kv"><span class="kk">Confidence</span><span class="vv" style="color:{conf_color(llm2['confidence_level'])}">{llm2['confidence_level']}</span></div>
      <div class="kv"><span class="kk">Data Quality</span><span class="vv">{llm2.get('data_quality','—')}</span></div>
      <div class="kv"><span class="kk">Explanation</span><span class="vv">{str(llm2.get('explanation','—'))[:250]}...</span></div>
      <div class="cg-div"></div>
      <div class="chain-label">CHAIN-OF-THOUGHT STEPS</div>
      {chain_html(llm2.get('reasoning_chain', []), '#ffb800')}
    </div>
  </div>

  <!-- LLM-3 -->
  <div class="cg-stage s3">
    <div class="cg-shead" onclick="this.nextElementSibling.classList.toggle('open')">
      <div><span class="cg-stag">LLM-3</span><span style="font-size:0.57rem;color:#4a6070;margin-left:8px;">PATTERN CLUSTERER</span></div>
      <span class="cg-sbadge" style="border-color:{conf_color(llm3['confidence_level'])};color:{conf_color(llm3['confidence_level'])}">{llm3['identified_pattern'].upper()} · {llm3['confidence_level']}</span>
    </div>
    <div class="cg-sbody">
      <div class="kv"><span class="kk">Pattern</span><span class="vv" style="color:#39ff14;text-transform:uppercase;">{llm3['identified_pattern']}</span></div>
      <div class="kv"><span class="kk">Confidence</span><span class="vv" style="color:{conf_color(llm3['confidence_level'])}">{llm3['confidence_level']}</span></div>
      <div class="kv"><span class="kk">Indicators</span><span class="vv">{', '.join(llm3.get('pattern_indicators', [])) or '—'}</span></div>
      <div class="cg-div"></div>
      <div class="chain-label">CHAIN-OF-THOUGHT STEPS</div>
      {chain_html(llm3.get('reasoning_chain', []), '#39ff14')}
    </div>
  </div>

  <!-- FUSION -->
  <div class="cg-stage sf">
    <div class="cg-shead" onclick="this.nextElementSibling.classList.toggle('open')">
      <div><span class="cg-stag">FUSION</span><span style="font-size:0.57rem;color:#4a6070;margin-left:8px;">WEIGHTED AGGREGATION</span></div>
      <span class="cg-sbadge" style="border-color:#ff4b6e;color:#ff4b6e">{fusion['agreement_score']} / 2.85</span>
    </div>
    <div class="cg-sbody open">
      <div class="kv"><span class="kk">Final Motivation</span><span class="vv" style="color:#ff4b6e;font-size:0.9rem;text-transform:uppercase;">{fusion['final_motivation']}</span></div>
      <div class="kv"><span class="kk">Agreement Score</span><span class="vv">{fusion['agreement_score']} / 2.85</span></div>
      <div class="kv"><span class="kk">Final Confidence</span><span class="vv" style="color:{conf_color(fusion['final_confidence'])}">{fusion['final_confidence']}</span></div>
      <div class="kv"><span class="kk">Models Agree</span><span class="vv" style="color:{'#39ff14' if fusion['models_agree'] else '#ff4b6e'}">{'✓ YES' if fusion['models_agree'] else '✗ NO — CONFLICT'}</span></div>
      <div class="kv"><span class="kk">Conflict</span><span class="vv" style="color:{'#ff4b6e' if fusion['conflict_detected'] else '#39ff14'}">{'⚠ YES' if fusion['conflict_detected'] else 'NONE'}</span></div>
      <div class="kv"><span class="kk">Recommendation</span><span class="vv">{fusion['recommendation']}</span></div>
    </div>
  </div>

  <!-- LLM-4 -->
  <div class="cg-stage s4">
    <div class="cg-shead" onclick="this.nextElementSibling.classList.toggle('open')">
      <div><span class="cg-stag">LLM-4</span><span style="font-size:0.57rem;color:#4a6070;margin-left:8px;">INTEGRATED FORENSIC REPORT</span></div>
      <span class="cg-sbadge" style="border-color:#00d4ff;color:#00d4ff">{llm4.get('status','—').upper()}</span>
    </div>
    <div class="cg-sbody open">
      <div class="cg-report">{llm4.get('report', 'No report generated.')}</div>
    </div>
  </div>

</div>

<script>
setTimeout(() => {{
  document.getElementById('cgb1').style.width = '{acc*100:.1f}%';
  document.getElementById('cgb2').style.width = '{f1*100:.1f}%';
  document.getElementById('cgb3').style.width = '{mae_bar:.1f}%';
  document.getElementById('cgb4').style.width = '{cols_bar:.1f}%';
  document.getElementById('cgb5').style.width = '{clust_bar:.1f}%';
  document.getElementById('cgb6').style.width = '{rec_bar:.1f}%';
  document.getElementById('cggauge').style.width = '{gauge_pct:.1f}%';
}}, 400);
</script>
"""
    display(HTML(html))

In [67]:
# ── Analyze a crime ───────────────────────────────────
crime = {
    "crime_text": "On 2020-05-10 at 22 hours, robbed a convenience store with handgun.",
    "crm_cd_desc": "robbery",
    "area_name": "downtown",
    "premis_desc": "store",
    "vict_age": "28",
    "vict_sex": "M",
    "weapon_desc": "handgun",
    "status_desc": "arrested",
    "domestic": "false"
}

result = analyze_crime_case(crime, verbose=False)

# ── Show GUI report ───────────────────────────────────
show_report(result, llm1_metadata, llm2_metadata, llm3_metadata)

⚠️ Gemini API error (attempt 1/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


⚠️ Gemini API error (attempt 2/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


❌ LLM-1 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 42.458925589s.


⚠️ Gemini API error (attempt 1/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


⚠️ Gemini API error (attempt 2/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


...

⚠️ Gemini API error (attempt 2/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


❌ LLM-3 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 32.303699341s.


⚠️ Gemini API error (attempt 1/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


⚠️ Gemini API error (attempt 2/3): 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%2


❌ LLM-4 failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 27.101619849s.

🧠 CHAIN-OF-THOUGHT REASONING (DETAILED)

────────────────────────────────────────────────────────────────────────────────
🔍 LLM-1: MOTIVATION ANALYSIS - CHAIN-OF-THOUGHT
────────────────────────────────────────────────────────────────────────────────
  ⚠️  No detailed reasoning chain returned

  📊 FINAL PREDICTION: UNKNOWN
  🎯 Confidence: Low
  💭 Summary: Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fl